<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_3/Lecture_3_1_%D0%90%D1%80%D1%85%D0%B8%D1%82%D0%B5%D0%BA%D1%82%D1%83%D1%80%D0%B0_Transformer_%E2%80%94_%D1%84%D1%83%D0%BD%D0%B4%D0%B0%D0%BC%D0%B5%D0%BD%D1%82_%D1%81%D0%BE%D0%B2%D1%80%D0%B5%D0%BC%D0%B5%D0%BD%D0%BD%D1%8B%D1%85_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.Введение к разделу «Архитектура Transformer»

### 1. Актуальность

За период с 2017 по 2024 год архитектура Transformer стала не просто доминирующей парадигмой в обработке естественного языка (NLP), но и универсальным фундаментом для широкого круга задач искусственного интеллекта. Предложенная в статье *«Attention Is All You Need»* [1], эта архитектура заменила рекуррентные и свёрточные слои исключительно механизмом внимания, что обеспечило беспрецедентную масштабируемость и эффективность обучения.

**Количественные показатели** убедительно свидетельствуют о влиянии Transformer. Оригинальная статья (NeurIPS 2017) по состоянию на 2025 год имеет более **120 000 цитирований**, входя в число наиболее цитируемых научных работ в области компьютерных наук. Количество публикаций, содержащих в названии или аннотации термин «Transformer», выросло с единиц в 2017 году до **более 15 000** в 2024 году (по данным arXiv). Число открытых и проприетарных моделей, основанных на этой архитектуре, превышает **несколько сотен**; наиболее известные из них — GPT-4, LLaMA, Qwen, Claude, Gemini, BERT, RoBERTa, T5 — составляют основу современного генеративного ИИ.

Важно подчеркнуть, что область применения Transformer не ограничивается текстом. Архитектура успешно адаптирована для:
- **Компьютерного зрения**: Vision Transformer (ViT) [2] и его производные демонстрируют результаты, сравнимые или превосходящие свёрточные сети (CNN) на эталонных наборах данных (ImageNet, COCO).
- **Обработки аудио**: Audio Spectrogram Transformer [3] и Whisper от OpenAI применяют Transformer для распознавания речи и классификации звуков.
- **Мультимодальных систем**: модели типа CLIP, Flamingo, GPT-4o используют Transformer для совместного представления текста, изображений, видео и аудио, открывая путь к агентному ИИ.

Таким образом, понимание архитектуры Transformer является **обязательным условием** для профессиональной деятельности в области машинного обучения, независимо от конкретной модальности данных.

---

### 2. Исторический контекст

До 2017 года обработка последовательностей (текстов, временных рядов, аудиосигналов) основывалась преимущественно на **рекуррентных нейронных сетях (RNN)** и их усовершенствованных вариантах — **LSTM**[4] и **GRU**[5]. Эти архитектуры обрабатывают данные последовательно, поддерживая скрытое состояние, которое переносится от одного шага к следующему. Основные ограничения такого подхода заключались в следующем:

1. **Последовательная обработка** не позволяет эффективно использовать современные GPU, что резко замедляет обучение на больших корпусах.
2. **Проблема дальних зависимостей** — информация из начальных токенов постепенно «затухает» при прохождении через многие шаги; LSTM лишь частично смягчает этот эффект.
3. В архитектурах **seq2seq** (энкодер–декодер), используемых для машинного перевода, энкодер сжимает всю входную последовательность в фиксированный вектор фиксированной размерности (контекстный вектор), что приводит к потере информации при работе с длинными предложениями.

Ключевым прорывом, предшествовавшим Transformer, стало введение **механизма внимания** в работе Bahdanau et al. (2014) [6]. Авторы предложили, чтобы декодер на каждом шаге генерации динамически выбирал, на какие части входной последовательности обращать внимание, вычисляя весовые коэффициенты на основе текущего состояния декодера и всех скрытых состояний энкодера. Это позволило существенно улучшить качество перевода, особенно для длинных предложений, и заложило основу для дальнейшего развития.

Однако в архитектуре Bahdanau внимание всё ещё сочеталось с RNN, сохраняя последовательную обработку. **Трансформация произошла в 2017 году**, когда команда Google Brain под руководством Ашвиша Васвани предложила полностью отказаться от рекуррентности, построив модель исключительно на слоях внимания. Результат превзошёл ожидания: новый подход не только достиг нового уровня точности на задачах машинного перевода, но и сократил время обучения с нескольких недель до нескольких дней благодаря полной параллелизации.

---

### 3. Цели раздела

Данный раздел ставит перед читателем следующие цели:

1. **Сформировать целостное представление об архитектуре Transformer** — от входного слоя до выходного, включая все промежуточные преобразования и потоки данных.

2. **Детально изучить каждый компонент**: механизм масштабированного точечного внимания (Scaled Dot-Product Attention), многоголовое внимание (Multi-Head Attention), позиционное кодирование (Positional Encoding), полносвязную сеть (Feed-Forward Network), остаточные связи и слои нормализации.

3. **Освоить математическую нотацию** и выводы формул, лежащих в основе Self-Attention:  
   \[
   \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V,
   \]
   а также понять, почему масштабирование на \(\sqrt{d_k}\) критически важно для стабильности градиентов.

4. **Разобраться в архитектурных вариациях** — различиях между энкодером, декодером и их комбинациями (Encoder-Only, Decoder-Only, Encoder-Decoder) и научиться обоснованно выбирать тип архитектуры для конкретной прикладной задачи.

5. **Получить практические навыки** реализации ключевых компонентов на Python (с использованием NumPy или PyTorch), что позволит в дальнейшем уверенно работать с библиотеками Hugging Face Transformers, адаптировать существующие модели и разрабатывать собственные.

---

### 4. Структура раздела

Изложение материала организовано по принципу «от общего к частному» и включает следующие темы:

- **От RNN к Transformer** — анализ ограничений рекуррентных архитектур, обоснование необходимости перехода к вниманию, ключевые результаты статьи 2017 года.
- **Общая архитектура Transformer** — структура энкодера и декодера, стек слоёв, поток данных; описание трёх основных вариантов (Encoder-Only, Decoder-Only, Encoder-Decoder) с примерами моделей.
- **Токенизация и эмбеддинги** — преобразование текста в последовательность идентификаторов, матрица эмбеддингов и её роль в преобразовании дискретных токенов в непрерывные векторные представления размерности \(d_{\text{model}}\).
- **Позиционное кодирование** — обоснование необходимости учёта порядка токенов; синусоидальные кодировки (фиксированные) и обучаемые позиционные эмбеддинги; современный подход Rotary Position Embedding (RoPE).
- **Механизм самовнимания (Self-Attention)** — интуиция, математическая формула, роль Query, Key, Value; влияние масштабирования.
- **Многоголовое внимание (Multi-Head Attention)** — объяснение, почему необходимо несколько независимых голов; параллельное вычисление, конкатенация и финальная проекция; современные эффективные вариации: Multi-Query Attention (MQA) и Grouped-Query Attention (GQA).
- **Полносвязная сеть (Feed-Forward Network, FFN)** — двухслойная структура, роль нелинейности; популярные функции активации (ReLU, GELU, SwiGLU) и их влияние на качество и вычислительную сложность.
- **Остаточные связи и нормализация** — принцип residual connections для устранения проблемы исчезающих градиентов; Layer Normalization и RMSNorm; сравнение Pre-Norm и Post-Norm, обоснование выбора Pre-Norm в современных реализациях.
- **Заключение и связь с последующими разделами** — резюме ключевых концепций; роль Transformer в тонкой настройке (LoRA), RAG-системах, агентах, RLHF и промышленном развёртывании.

Материал раздела обеспечивает необходимый фундамент для всех последующих глав: глубокое понимание архитектуры является предпосылкой для эффективной работы с предобученными моделями, их адаптации, оценки и эксплуатации.

---

### 5. Визуализация архитектуры

Приведённая ниже диаграмма (в нотации Mermaid) отображает полную структуру Transformer в конфигурации энкодер–декодер, соответствующей оригинальной статье. Она включает все основные блоки и потоки данных, что позволяет читателю визуально связать теоретические описания с реальными компонентами.

```mermaid
flowchart TD
    subgraph Input["1. Входные данные"]
        S[Входная последовательность<br/>токенов длины T]
    end

    subgraph Embedding["2. Слой эмбеддингов"]
        E[Матрица эмбеддингов<br/>размером T×d_model]
    end

    subgraph PE["3. Позиционное кодирование"]
        P[Сложение эмбеддингов<br/>с позиционными кодировками]
    end

    subgraph EncoderStack["4. Стек энкодеров (N×)"]
        direction TB
        EN1[Энкодер слой 1]
        EN2[Энкодер слой 2]
        ENDot[⋮]
        ENN[Энкодер слой N]
    end

    subgraph EncoderLayer["Структура слоя энкодера"]
        direction TB
        MHA1[Multi-Head<br/>Self-Attention]
        AD1[Add & LayerNorm]
        FFN1[Feed-Forward<br/>Network]
        AD2[Add & LayerNorm]
    end

    subgraph DecoderStack["5. Стек декодеров (N×)"]
        direction TB
        DN1[Декодер слой 1]
        DN2[Декодер слой 2]
        DNDot[⋮]
        DNN[Декодер слой N]
    end

    subgraph DecoderLayer["Структура слоя декодера"]
        direction TB
        MMHA[Masked Multi-Head<br/>Self-Attention]
        AD3[Add & LayerNorm]
        CMHA[Cross-Attention<br/>(Q из декодера, K,V из энкодера)]
        AD4[Add & LayerNorm]
        FFN2[Feed-Forward<br/>Network]
        AD5[Add & LayerNorm]
    end

    subgraph Output["6. Выходной слой"]
        LC[Линейный слой<br/>(d_model → vocab_size)]
        SM[Softmax<br/>вероятности токенов]
    end

    S --> E --> P --> EncoderStack
    EncoderStack -->|выход энкодера| DecoderStack

    P --> MHA1 --> AD1 --> FFN1 --> AD2
    AD2 --> EN2

    AD2 -->|вход в декодер| MMHA --> AD3 --> CMHA --> AD4 --> FFN2 --> AD5
    AD5 --> DN2

    ENN -->|K, V| CMHA

    DNN --> LC --> SM --> O[Выходная последовательность<br/>токенов]

    style Input fill:#e3f2fd,stroke:#1565c0
    style Embedding fill:#e8f5e9,stroke:#2e7d32
    style PE fill:#fff3e0,stroke:#e65100
    style EncoderStack fill:#f3e5f5,stroke:#6a1b9a
    style DecoderStack fill:#fce4ec,stroke:#c62828
    style Output fill:#e0f7fa,stroke:#00838f
```

---

### 6. Ключевые термины

| Термин | Определение |
|--------|-------------|
| **Transformer** | Архитектура нейронной сети, основанная исключительно на механизме самовнимания и полносвязных слоях, без рекуррентных или свёрточных компонентов. |
| **Self-Attention** | Механизм, позволяющий каждому элементу последовательности взаимодействовать со всеми остальными, вычисляя весовые коэффициенты на основе их сходства. |
| **Multi-Head Attention** | Параллельное выполнение нескольких независимых функций внимания с последующей конкатенацией и линейным преобразованием. |
| **Query, Key, Value (Q, K, V)** | Три векторных представления, получаемые линейным проецированием входных данных; используются для вычисления внимания. |
| **Scaled Dot-Product Attention** | Основная формула внимания: \(\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V\), где \(d_k\) — размерность ключей. |
| **Positional Encoding** | Добавление к эмбеддингам информации о позиции токена, чтобы модель могла учитывать порядок элементов. |
| **Encoder** | Часть архитектуры, преобразующая входную последовательность в непрерывное контекстное представление. |
| **Decoder** | Часть архитектуры, генерирующая выходную последовательность на основе представления энкодера и ранее сгенерированных токенов. |
| **Masked Self-Attention** | Вариант самовнимания в декодере, где будущим токенам запрещено влиять на текущий (маска из −∞). |
| **Cross-Attention** | Внимание, где запросы (Q) берутся из декодера, а ключи (K) и значения (V) — из выхода энкодера. |
| **Feed-Forward Network (FFN)** | Двухслойная полносвязная сеть, применяемая позиционно-независимо к каждому элементу последовательности. |
| **Residual Connection** | Связь, добавляющая вход подслоя к его выходу: \(x_{\text{out}} = x + \text{Sublayer}(x)\), облегчающая обучение глубоких сетей. |
| **Layer Normalization** | Нормализация активаций по признаковому измерению для каждой позиции отдельно; стабилизирует обучение. |
| **Pre-Norm / Post-Norm** | Схемы размещения нормализации: до (pre-norm) или после (post-norm) подслоя; современные модели используют pre-norm. |

---

### 7. Рекомендуемая литература

**Основная**
1. Vaswani, A., et al. (2017). *«Attention Is All You Need»*. Advances in Neural Information Processing Systems (NeurIPS).  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

**Для углублённого изучения**
2. Alammar, J. (2018). *«The Illustrated Transformer»*. — Визуальное, интуитивное объяснение архитектуры.  
   🔗 [https://jalammar.github.io/illustrated-transformer/](https://jalammar.github.io/illustrated-transformer/)

3. Phuong, M., & Hutter, M. (2022). *«Formal Algorithms for Transformers»*. arXiv:2207.09238. — Формальное описание алгоритмов Transformer.  
   🔗 [https://arxiv.org/abs/2207.09238](https://arxiv.org/abs/2207.09238)

4. Liu, Y., et al. (2024). *«Recent Progress in Understanding Transformer and Developing Its Surpasser: A Survey»*. — Обзор современных достижений в понимании и развитии Transformer.  
   🔗 [https://ieeexplore.ieee.org/document/10878057](https://ieeexplore.ieee.org/document/10878057)

**Обзорные статьи**
5. Sajun, A.R., et al. (2024). *«A Historical Survey of Advances in Transformer Architectures»*. Applied Sciences, 14(10), 4316. — Исторический обзор развития Transformer-архитектур.  
   🔗 [https://doi.org/10.3390/app14104316](https://doi.org/10.3390/app14104316)

6. Shao, M., et al. (2024). *«Survey of Different Large Language Model Architectures: Trends, Benchmarks, and Challenges»*. IEEE Access. — Обзор LLM-архитектур на основе Transformer.  
   🔗 [https://arxiv.org/abs/2412.03454](https://arxiv.org/abs/2412.03454)

**Практические ресурсы**
7. Harvard NLP. *«The Annotated Transformer»*. — Пошаговая реализация Transformer на PyTorch.  
   🔗 [https://nlp.seas.harvard.edu/2018/04/03/attention.html](https://nlp.seas.harvard.edu/2018/04/03/attention.html)

8. Hugging Face. *«NLP Course»*. — Практические занятия с библиотекой Transformers.  
   🔗 [https://huggingface.co/learn/nlp-course](https://huggingface.co/learn/nlp-course)

---

Понимание архитектуры Transformer является не просто ознакомлением с очередной моделью, а освоением ключевого принципа, на котором строится современный искусственный интеллект. Каждый компонент этой архитектуры — от масштабированного произведения до слоя нормализации — был разработан для решения конкретной инженерной задачи, и их совокупность обеспечивает невиданную ранее масштабируемость и универсальность. Изучение этого раздела закладывает основу для всех последующих тем курса, от тонкой настройки до промышленного развёртывания систем на основе больших языковых моделей.

## Тема 1.1. Ограничения рекуррентных нейронных сетей (RNN)

Рекуррентные нейронные сети долгое время являлись стандартным инструментом для обработки последовательных данных, однако их фундаментальные ограничения стали основным препятствием на пути к созданию моделей, способных эффективно работать с длинными зависимостями и большими объёмами данных. Понимание этих ограничений необходимо для осознания причин появления архитектуры Transformer, которая полностью устраняет рекуррентные связи.

---

### 1. Математическое описание RNN и принцип работы

Стандартная RNN (в конфигурации «Элмана») описывается рекуррентным уравнением, связывающим текущее скрытое состояние $h_t$ с предыдущим состоянием $h_{t-1}$ и текущим входом $x_t$:

$$
h_t = f\bigl(W_h \, h_{t-1} + W_x \, x_t + b\bigr),
\tag{1}
$$

где:

- $h_t \in \mathbb{R}^{d}$ — скрытое состояние в момент времени $t$,
- $x_t \in \mathbb{R}^{m}$ — входной вектор (например, эмбеддинг токена),
- $W_h \in \mathbb{R}^{d \times d}$ — матрица рекуррентных весов,
- $W_x \in \mathbb{R}^{d \times m}$ — матрица входных весов,
- $b \in \mathbb{R}^{d}$ — вектор смещений,
- $f$ — нелинейная функция активации (обычно $\tanh$ или $\text{ReLU}$).

Выходной сигнал на каждом шаге обычно вычисляется как $y_t = g(W_y \, h_t + b_y)$, где $g$ — функция активации для выходного слоя (например, softmax для классификации).

Развёртка RNN по времени представляет собой цепочку одинаковых преобразований, где каждая копия использует одни и те же весовые матрицы. Это позволяет представить обработку последовательности длины $T$ как глубокую сеть с $T$ слоями, но с общей весовой матрицей. Такая структура показана на диаграмме:

```mermaid
flowchart LR
    subgraph time["Развёртка по времени"]
        x1[x₁] --> h1[("h₁")]
        h0[("h₀")] --> h1
        h1 --> y1[y₁]
        
        x2[x₂] --> h2[("h₂")]
        h1 --> h2
        h2 --> y2[y₂]
        
        xT[x_T] --> hT[("h_T")]
        hTminus1[("h_{T-1}")] --> hT
        hT --> yT[y_T]
    end
    
    style h1 fill:#bbdefb
    style h2 fill:#bbdefb
    style hT fill:#bbdefb
```

Обучение RNN осуществляется с помощью **обратного распространения ошибки во времени (Backpropagation Through Time, BPTT)**. Функция потерь обычно определяется как сумма потерь на каждом временном шаге:

$$
\mathcal{L} = \sum_{t=1}^{T} \mathcal{L}_t\bigl(y_t, \hat{y}_t\bigr).
$$

Градиент по отношению к рекуррентной матрице $W_h$ вычисляется как сумма вкладов от каждого шага:

$$
\frac{\partial \mathcal{L}}{\partial W_h}
= \sum_{t=1}^{T}
\frac{\partial \mathcal{L}_t}{\partial h_t}
\cdot \left( \prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}} \right)
\cdot \frac{\partial h_k}{\partial W_h}.
\tag{2}
$$

Произведение якобианов $\prod \frac{\partial h_i}{\partial h_{i-1}}$ является ключевым фактором, определяющим стабильность обучения. Каждый якобиан имеет вид

$$
\frac{\partial h_i}{\partial h_{i-1}}
= \mathrm{diag}\bigl(f'(W_h h_{i-1} + W_x x_i + b)\bigr) \cdot W_h.
\tag{3}
$$

Поскольку производная активации $f'$ ограничена по модулю (для $\tanh$ она лежит в интервале $(0,1]$), а спектральные свойства матрицы $W_h$ определяют, будут ли последовательные умножения приводить к экспоненциальному затуханию или росту, возникает фундаментальная нестабильность градиентов.

---

### 2. Три главных ограничения RNN

#### 2.1. Последовательная обработка и отсутствие параллелизации

Из уравнения (1) видно, что состояние $h_t$ не может быть вычислено до получения $h_{t-1}$. Это накладывает жёсткое ограничение на параллелизацию: все шаги должны выполняться последовательно, что делает невозможным использование современных графических процессоров, оптимизированных для массовых параллельных вычислений.

Вычислительная сложность обработки последовательности длины $T$ составляет

$$
\mathcal{C}_{\text{RNN}} = O\left(T \cdot (d^2 + d \cdot m)\right),
\tag{4}
$$

где $d$ — размерность скрытого состояния, а $m$ — размерность входа. Время обработки растёт линейно с длиной последовательности, и для длинных текстов (тысячи токенов) это становится неприемлемо медленным. Кроме того, для реализации BPTT необходимо хранить все промежуточные состояния $h_0, h_1, \dots, h_T$, что даёт сложность по памяти $O(T \cdot d)$. Это ограничивает как максимальную длину последовательности, так и размерность модели, которую можно обучить на доступном оборудовании.

Таким образом, последовательная природа RNN является прямым противоречием с архитектурой современных вычислительных систем, что делает масштабирование RNN на большие данные крайне неэффективным.

#### 2.2. Исчезающие и взрывающиеся градиенты

Второе, и, возможно, наиболее серьёзное ограничение связано с нестабильностью градиентов при обратном распространении через длинные последовательности. Как показано в выражении (2), градиент содержит произведение якобианов $\prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}}$. Каждый якобиан (3) представляет собой произведение диагональной матрицы производных активации и матрицы $W_h$.

Если спектральный радиус $\rho(W_h)$ меньше единицы (с учётом ограниченности производных активации), то произведение таких матриц экспоненциально убывает с ростом $t-k$, что приводит к **исчезающим градиентам**. В этом случае градиенты для ранних шагов становятся пренебрежимо малыми, и модель не может обновлять веса, отвечающие за долгосрочные зависимости. Это проявляется в том, что RNN «забывает» информацию из начала последовательности, что делает её бесполезной для задач, требующих учёта контекста на большом расстоянии (например, анализ длинных документов или диалогов).

Напротив, если $\rho(W_h) > 1$, то градиенты экспоненциально растут, вызывая **взрывающиеся градиенты**, которые приводят к численной нестабильности и резким скачкам функции потерь. Взрывающиеся градиенты можно частично контролировать с помощью клиппинга (ограничения нормы градиента), однако исчезающие градиенты не имеют простого решения, так как они связаны с самой структурой рекуррентных связей.

На диаграмме ниже схематично показано, как градиент затухает при обратном распространении через длинную последовательность:

```mermaid
flowchart LR
    subgraph forward["Прямой проход"]
        hT["h_T"] --> hTminus["h_{T-1}"] --> hTminus2["h_{T-2}"] --> hTminus3["..."]
    end

    subgraph backward["Обратный проход (BPTT)"]
        direction LR
        gT["градиент у h_T"] -->|"× J_T"| gTminus["градиент у h_{T-1}"]
        gTminus -->|"× J_{T-1}"| gTminus2["градиент у h_{T-2}"]
        gTminus2 -->|"× J_{T-2}"| gTminus3["..."]
        gTminus3 -->|"малый градиент"| g0["почти нулевой градиент у h_0"]
    end

    style gT fill:#ffcdd2
    style gTminus fill:#ffcdd2
    style gTminus2 fill:#ffcdd2
    style gTminus3 fill:#ffcdd2
    style g0 fill:#e0e0e0
```

#### 2.3. Ограниченная память («бутылочное горлышко»)

Третье фундаментальное ограничение заключается в том, что скрытое состояние $h_t$ имеет **фиксированную размерность** $d$, независимо от длины последовательности. Это означает, что вся информация о предыдущих $t-1$ элементах должна быть сжата в вектор фиксированной длины. Такое сжатие неизбежно приводит к потере информации, особенно при работе с длинными последовательностями, содержащими множество разнородных фактов.

Это явление часто называют **«бутылочным горлышком»** информационного канала: модель вынуждена выбирать, какую информацию сохранять, а какую отбрасывать. В отличие от архитектур, позволяющих хранить информацию в отдельных векторах для каждого элемента (как, например, в механизме внимания), RNN не может сохранять детализированную информацию о всех предшествующих элементах, поскольку они должны быть «спроецированы» в одно состояние. Это ограничение особенно критично для задач, требующих одновременного учёта множества независимых фактов, таких как чтение больших документов или поддержание диалога.

---

### 3. Попытки решения: LSTM и GRU

Для преодоления проблемы исчезающих градиентов были разработаны усовершенствованные рекуррентные архитектуры — **долгая краткосрочная память (LSTM)** [[1](https://www.bioinf.jku.at/publications/older/2604.pdf)] и **управляемые рекуррентные блоки (GRU)** [[2](https://arxiv.org/abs/1406.1078)]. Основная идея заключается во введении **механизмов ворот**, которые управляют потоком информации, позволяя избирательно сохранять или забывать информацию на длительные промежутки времени.

**LSTM** содержит три типа ворот: входные, забывающие и выходные. Обновление состояния описывается следующими уравнениями:

$$
\begin{aligned}
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i), \\
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f), \\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o), \\
\tilde{C}_t &= \tanh(W_c x_t + U_c h_{t-1} + b_c), \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t, \\
h_t &= o_t \odot \tanh(C_t),
\end{aligned}
$$

где $\odot$ — поэлементное умножение, $C_t$ — состояние ячейки, а $i_t, f_t, o_t$ — активации ворот. Забывающий вентиль $f_t$ позволяет модели сбрасывать информацию из ячейки, а входной вентиль $i_t$ — добавлять новую. Благодаря этому градиенты могут проходить через ячейку без изменений, если $f_t \approx 1$, что существенно смягчает проблему исчезающих градиентов.

**GRU** является упрощённой версией LSTM, объединяющей входной и забывающий вентили в один «вентиль обновления»:

$$
\begin{aligned}
z_t &= \sigma(W_z x_t + U_z h_{t-1} + b_z), \\
r_t &= \sigma(W_r x_t + U_r h_{t-1} + b_r), \\
\tilde{h}_t &= \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h), \\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t.
\end{aligned}
$$

Эти архитектуры действительно улучшают способность модели к запоминанию долгосрочных зависимостей, однако они **не устраняют** основные ограничения RNN:

- **Последовательная обработка** остаётся неизменной: для вычисления $h_t$ необходимо дождаться $h_{t-1}$, что не позволяет распараллеливать вычисления.
- **Фиксированная размерность** скрытого состояния сохраняется, и «бутылочное горлышко» никуда не исчезает.
- Даже с LSTM обучение на очень длинных последовательностях (тысячи шагов) остаётся сложным, и на практике LSTM редко работают устойчиво при длинах более 200–500 шагов.

Сравнительная характеристика трёх архитектур приведена в таблице:

| Характеристика | RNN | LSTM | GRU |
|----------------|-----|------|-----|
| Число параметров (на шаг) | $d^2 + d \cdot m$ | $4(d^2 + d \cdot m)$ | $3(d^2 + d \cdot m)$ |
| Сложность обучения (на шаг) | $O(d^2)$ | $O(d^2)$ | $O(d^2)$ |
| Способность к дальним зависимостям | Очень ограничена | Улучшена (до сотен шагов) | Улучшена (до сотен шагов) |
| Параллелизация | Нет | Нет | Нет |
| Проблема «бутылочного горлышка» | Присутствует | Присутствует | Присутствует |

---

### 4. Пример задачи, где RNN терпят неудачу

Рассмотрим задачу **извлечения отношений** из длинных документов. Пусть дано предложение:

*«Альберт Эйнштейн, родившийся в Ульме в 1879 году и получивший Нобелевскую премию в 1921 году за открытие фотоэлектрического эффекта, позже эмигрировал в США и принял гражданство в 1940 году.»*

Требуется ответить на вопрос: *«В каком году Эйнштейн стал гражданином США?»*. Ответ (1940) находится в самом конце предложения, а ключевой субъект «Эйнштейн» — в начале. Для RNN, даже с LSTM, информация о субъекте должна быть сохранена на протяжении более 30 слов, что уже близко к пределу возможностей LSTM. На более длинных текстах (например, несколько абзацев) RNN практически гарантированно теряют связь между субъектом и предикатом, порождая ошибочные ответы или галлюцинации.

Именно такие сценарии продемонстрировали, что рекуррентные подходы неспособны обеспечить надёжное моделирование долгосрочных зависимостей, которые естественно возникают в языке, биологических последовательностях и других структурированных данных.

---

### 5. Выводы и требования к новой архитектуре

Анализ ограничений RNN приводит к формулировке требований к архитектуре, которая могла бы преодолеть эти фундаментальные препятствия:

1. **Параллелизуемость**: вычисления для всех элементов последовательности должны выполняться одновременно, без зависимости от порядка. Это требует отказа от рекуррентных связей.

2. **Прямые связи между произвольными позициями**: модель должна обеспечивать возможность непосредственного взаимодействия между любыми двумя токенами последовательности с постоянной вычислительной сложностью, не зависящей от расстояния между ними.

3. **Отсутствие ограничения на длину памяти**: архитектура не должна требовать сжатия всей истории в вектор фиксированной размерности; каждое представление должно сохранять свою индивидуальность и доступность.

4. **Стабильность градиентов**: распространение градиентов должно быть устойчивым и не зависеть от длины последовательности, желательно путём устранения рекуррентных произведений якобианов.

Именно эти требования легли в основу разработки архитектуры **Transformer**, которая заменила рекуррентные связи механизмом самовнимания (self-attention), обеспечивающим прямые взаимодействия между всеми парами элементов последовательности. Это позволило достичь полной параллелизации, постоянной сложности связи произвольных позиций и стабильного распространения градиентов, что в итоге привело к революционным результатам в области обработки естественного языка и других модальностей.

---

### Литература

1. Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural computation*, 9(8), 1735-1780.  
   🔗 [https://www.bioinf.jku.at/publications/older/2604.pdf](https://www.bioinf.jku.at/publications/older/2604.pdf)

2. Cho, K., van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H., & Bengio, Y. (2014). Learning phrase representations using RNN encoder-decoder for statistical machine translation. *arXiv preprint arXiv:1406.1078*.  
   🔗 [https://arxiv.org/abs/1406.1078](https://arxiv.org/abs/1406.1078)

3. Bengio, Y., Simard, P., & Frasconi, P. (1994). Learning long-term dependencies with gradient descent is difficult. *IEEE Transactions on Neural Networks*, 5(2), 157-166.  
   🔗 [https://ieeexplore.ieee.org/document/279181](https://ieeexplore.ieee.org/document/279181)

4. Pascanu, R., Mikolov, T., & Bengio, Y. (2013). On the difficulty of training recurrent neural networks. *Proceedings of the 30th International Conference on Machine Learning (ICML)*.  
   🔗 [https://arxiv.org/abs/1211.5063](https://arxiv.org/abs/1211.5063)

## Тема 1.2. Статья «Attention Is All You Need»: рождение новой парадигмы

В декабре 2017 года на конференции NeurIPS была представлена работа, которой суждено было изменить траекторию развития искусственного интеллекта. Статья под названием **«Attention Is All You Need»** [1], подготовленная исследовательской группой Google Brain, предложила архитектуру, отказавшуюся от доминировавших на тот момент рекуррентных и свёрточных слоёв в пользу исключительно механизма внимания. Этот текст представляет собой первую публикацию, в которой была описана архитектура **Transformer**, ставшая фундаментом современных больших языковых моделей.

---

### 1. Информация о статье

**Полное название:** *Attention Is All You Need*

**Авторы:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin. Восемь исследователей из Google Brain и смежных организаций, объединивших усилия вокруг идеи, которая впоследствии определила развитие целой области.

**Год публикации:** 2017

**Конференция:** NeurIPS (Advances in Neural Information Processing Systems) — одна из наиболее престижных международных конференций в области машинного обучения.

**Ключевая идея:** авторы предложили *«новую простую сетевую архитектуру, Transformer, основанную исключительно на механизмах внимания, полностью отказывающуюся от рекуррентности и свёрток»*【3†L3-L5】. Это был радикальный шаг: вместо того чтобы улучшать существующие рекуррентные подходы, исследователи предложили заменить их принципиально иной парадигмой.

---

### 2. Контекст появления

До 2017 года доминирующими архитектурами для задач машинного перевода и моделирования последовательностей были рекуррентные нейронные сети (RNN) и их усовершенствованные варианты — LSTM и GRU. Как отмечают сами авторы во введении, *«рекуррентные нейронные сети, в особенности LSTM и GRU, прочно утвердились как современные подходы в моделировании последовательностей и задачах транзакции, таких как языковое моделирование и машинный перевод»*【1†L25-L27】.

На тот момент лучшие модели машинного перевода представляли собой сложные архитектуры на основе RNN с механизмом внимания, который соединял энкодер и декодер. Среди них выделялись:

- **GNMT** (Google Neural Machine Translation) — система Google, достигшая на тот момент впечатляющих результатов【7†L15-L18】;
- **ConvS2S** — модель на основе свёрточных слоёв от Facebook【7†L15-L18】;
- **Deep-Att + PosUnk** — глубокая модель с вниманием и обработкой неизвестных слов【7†L15-L18】.

Однако все эти подходы обладали фундаментальными ограничениями, которые становились всё более очевидными по мере роста объёмов данных и требований к качеству:

1. **Последовательная обработка.** RNN обрабатывают токены один за другим, что делает невозможной параллелизацию вычислений и резко замедляет обучение на больших корпусах.

2. **Проблема дальних зависимостей.** Информация из начала последовательности постепенно «размывается» к концу, и даже LSTM не полностью решают эту проблему.

3. **Огромные вычислительные затраты.** Обучение лучших моделей машинного перевода того времени требовало недель на множестве GPU и стоило миллионы долларов.

Именно эти нерешённые проблемы создали почву для появления принципиально новой архитектуры.

---

### 3. Ключевые инновации

Статья предложила четыре основные инновации, которые в совокупности определили успех Transformer.

#### 3.1. Scaled Dot-Product Attention

Авторы предложили модификацию механизма внимания, названную *«масштабированное точечное внимание»* (Scaled Dot-Product Attention). Его суть описывается формулой:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Ключевое новшество — **масштабирование на** $\sqrt{d_k}$. Авторы объясняют это так: *«Мы подозреваем, что для больших значений $d_k$ скалярные произведения сильно растут по величине, заталкивая функцию softmax в области с крайне малыми градиентами. Чтобы противодействовать этому эффекту, мы масштабируем скалярные произведения на $1/\sqrt{d_k}$»*【4†L30-L35】.

Это решение оказалось критически важным для стабильности обучения: без масштабирования градиенты становились слишком малыми, и модель не могла эффективно обучаться.

#### 3.2. Multi-Head Attention

Вместо выполнения одной функции внимания авторы предложили **многоголовое внимание** (Multi-Head Attention). Идея заключается в том, чтобы линейно спроецировать запросы, ключи и значения $h$ раз с различными обучаемыми проекциями, выполнить внимание параллельно на каждой проекции, а затем объединить результаты.

Формально это записывается так:

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O,
$$

где

$$
\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V).
$$

Авторы поясняют мотивацию: *«Многоголовое внимание позволяет модели совместно обращать внимание на информацию из разных подпространств представлений на разных позициях. При одной голове внимания усреднение препятствует этому»*【4†L38-L40】. В работе использовалось $h = 8$ параллельных голов внимания с размерностью $d_k = d_v = d_{\text{model}} / h = 64$.

#### 3.3. Positional Encoding

Поскольку в модели нет ни рекуррентности, ни свёрток, она не имеет встроенного понимания порядка токенов. Для решения этой проблемы авторы ввели **позиционное кодирование** (Positional Encoding) — информацию о позиции токена в последовательности, которая добавляется к эмбеддингам на входе энкодера и декодера.

В работе используются синусоидальные функции разных частот:

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right),
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right).
$$

Авторы объясняют выбор: *«Мы выбрали эту функцию, потому что предположили, что она позволит модели легко обучаться обращать внимание по относительным позициям, поскольку для любого фиксированного смещения $k$ позиционное кодирование $PE_{pos+k}$ может быть представлено как линейная функция от $PE_{pos}$»*【6†L30-L35】.

#### 3.4. Отказ от рекуррентности

Самый радикальный шаг — полный отказ от рекуррентных и свёрточных слоёв. Вместо них архитектура строится исключительно на:

- **Self-Attention** в энкодере — каждый токен взаимодействует со всеми токенами;
- **Masked Self-Attention** в декодере — запрет на обращение к будущим токенам для сохранения авторегрессивности;
- **Cross-Attention** — связь между энкодером и декодером.

Это решение обеспечило **полную параллелизацию** вычислений и **постоянное число операций** для связи любых двух позиций, в отличие от $O(n)$ у RNN【6†L5-L10】.

---

### 4. Экспериментальные результаты

#### 4.1. Качество перевода

Результаты оказались впечатляющими. На задаче перевода с английского на немецкий (WMT 2014 English-German) Transformer достиг **BLEU-оценки 28.4**, превзойдя лучшие ранее опубликованные результаты, включая ансамбли, *«более чем на 2 BLEU»*【3†L5-L8】.

На задаче перевода с английского на французский (WMT 2014 English-French) модель установила *«новый современный результат для одной модели — 41.0 BLEU»*【3†L5-L8】.

Для сравнения, лучшие модели того времени показывали следующие результаты:

| Модель | EN-DE BLEU | EN-FR BLEU |
|--------|------------|------------|
| ByteNet | 23.75 | — |
| GNMT + RL | 24.6 | 39.92 |
| ConvS2S | 25.16 | 40.46 |
| MoE | 26.03 | 40.56 |
| **Transformer (base)** | **27.3** | **38.1** |
| **Transformer (big)** | **28.4** | **41.0** |

*Источник: таблица 2 из оригинальной статьи*【8†L5-L20】.

#### 4.2. Время обучения

Ещё более впечатляющим оказалось сокращение времени обучения. Базовая модель обучалась **12 часов** на 8 GPU P100. Большая модель — **3.5 дня** на тех же 8 GPU【7†L15-L18】.

Для сравнения, лучшие RNN-модели того времени требовали нескольких недель обучения на сопоставимом или большем количестве GPU. Авторы подчёркивают: *«Даже наша базовая модель превосходит все ранее опубликованные модели и ансамбли при доле вычислительных затрат любой из конкурирующих моделей»*【8†L22-L25】.

```mermaid
xychart-beta
    title "Время обучения (в днях) — сравнение моделей"
    x-axis ["RNN-based", "ConvS2S", "Transformer (base)", "Transformer (big)"]
    y-axis "Дни" 0 --> 25
    bar [21, 14, 0.5, 3.5]
```

#### 4.3. Сравнение сложности слоёв

Авторы также провели систематическое сравнение различных типов слоёв по трём критериям: вычислительная сложность, возможность параллелизации и длина пути для дальних зависимостей【6†L5-L10】.

| Тип слоя | Сложность на слой | Минимальное число последовательных операций | Максимальная длина пути |
|----------|-------------------|---------------------------------------------|-------------------------|
| Self-Attention | $O(n^2 \cdot d)$ | $O(1)$ | $O(1)$ |
| Recurrent | $O(n \cdot d^2)$ | $O(n)$ | $O(n)$ |
| Convolutional | $O(k \cdot n \cdot d^2)$ | $O(1)$ | $O(\log_k(n))$ |

*Источник: таблица 1 из оригинальной статьи*【6†L5-L10】.

Как видно из таблицы, Self-Attention обеспечивает **постоянное число операций** (полная параллелизация) и **минимальную длину пути** для связи любых двух позиций, что критически важно для обучения дальним зависимостям.

---

### 5. Влияние на поле

Влияние статьи «Attention Is All You Need» на развитие искусственного интеллекта трудно переоценить. Она стала катализатором, запустившим цепную реакцию прорывов.

#### 5.1. Появление BERT и GPT

В 2018 году, менее чем через год после публикации, появился **BERT** (Bidirectional Encoder Representations from Transformers) от Google — модель, использующая только энкодер Transformer и установившая новые рекорды на 11 NLP-бенчмарках. BERT продемонстрировал, что предобучение на больших корпусах с последующей тонкой настройкой даёт беспрецедентные результаты.

Параллельно **OpenAI** развивала декодерную ветвь Transformer. В 2018 году вышел **GPT-1**, в 2019 — **GPT-2** (1.5 млрд параметров), а в 2020 — **GPT-3** (175 млрд параметров). Каждая новая итерация подтверждала масштабируемость Transformer: увеличение размера модели и данных вело к появлению новых, эмерджентных способностей.

#### 5.2. Доминирование Transformer в NLP

К 2024 году Transformer стал универсальным стандартом в обработке естественного языка. Все ведущие модели — GPT-4, Claude, LLaMA, Qwen, Gemini — построены на этой архитектуре. Оригинальная статья была процитирована более **120 000 раз**, войдя в число наиболее цитируемых научных работ в истории компьютерных наук.

#### 5.3. Выход за пределы NLP

Важно подчеркнуть, что влияние Transformer вышло далеко за рамки текстовых задач. Архитектура была успешно адаптирована для:

- **Компьютерного зрения** — Vision Transformer (ViT) показал результаты, сравнимые с лучшими свёрточными сетями;
- **Обработки аудио** — Audio Spectrogram Transformer и Whisper;
- **Мультимодальных систем** — CLIP, Flamingo, GPT-4o объединяют текст, изображения, видео и аудио.

Как отмечают сами авторы в заключении: *«Мы с воодушевлением смотрим в будущее моделей, основанных на внимании, и планируем применить их к другим задачам. Мы планируем расширить Transformer на задачи, включающие другие модальности ввода и вывода, помимо текста»*【9†L22-L25】.

#### 5.4. Цитаты, изменившие мир

В статье есть несколько фраз, которые стали программными для целого поколения исследователей:

> *«Мы предлагаем новую простую сетевую архитектуру, Transformer, основанную исключительно на механизмах внимания, полностью отказывающуюся от рекуррентности и свёрток»*【3†L3-L5】.

> *«Внимание — это всё, что вам нужно»* — сам заголовок статьи стал манифестом новой эпохи.

---

### 6. Заключение

Статья «Attention Is All You Need» — это редкий пример работы, которая не просто улучшила существующие методы, а **переопределила направление развития целой области**. Отказ от рекуррентности в пользу внимания оказался не просто техническим решением, а сменой парадигмы, позволившей масштабировать модели до невиданных ранее размеров и открывшей путь к созданию систем, которые сегодня мы называем искусственным интеллектом общего назначения.

Transformer стал не просто архитектурой — он стал **фундаментом**, на котором строится современный ИИ. Понимание этой архитектуры — ключ к пониманию того, как работают современные языковые модели, и необходимое условие для участия в создании следующего поколения интеллектуальных систем.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. Advances in Neural Information Processing Systems (NeurIPS).  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 1.3. Преимущества Transformer: новая парадигма обработки последовательностей

Архитектура Transformer, предложенная в статье «Attention Is All You Need» [1], принесла с собой не просто улучшение качества, а принципиально новый способ обработки последовательных данных. Её преимущества перед рекуррентными и свёрточными архитектурами столь значительны, что в течение нескольких лет она вытеснила все предыдущие подходы, став стандартом де-факто в области искусственного интеллекта. В этом разделе мы систематически рассмотрим четыре ключевых преимущества Transformer: **параллелизуемость**, **глобальный контекст**, **масштабируемость** и **универсальность**.

---

### 1. Параллелизация: обработка всех токенов одновременно

Фундаментальное ограничение рекуррентных нейронных сетей заключается в их последовательной природе: для вычисления скрытого состояния на шаге $t$ необходимо дождаться завершения вычислений на шаге $t-1$. Это приводит к тому, что общее время обработки последовательности длины $T$ пропорционально $T$, и использование параллельных вычислений на GPU оказывается крайне ограниченным.

В отличие от RNN, механизм самовнимания (Self-Attention) позволяет вычислять представление каждого токена независимо от остальных, используя для этого все позиции одновременно. Математически это выражается в том, что матричное произведение $QK^T$ вычисляется за один шаг с использованием высокооптимизированных библиотек линейной алгебры, которые эффективно задействуют все ядра GPU.

Сравнение вычислительной сложности демонстрирует это различие. Для последовательности длины $T$ и размерности представления $d$:

- **RNN**: каждый шаг требует $O(d^2)$ операций (умножение на матрицу $W_h$), и таких шагов $T$, следовательно, общая сложность $O(T \cdot d^2)$.
- **Transformer**: основная операция — умножение матриц $QK^T$ размера $(T \times d)$ на $(d \times T)$, что требует $O(T^2 \cdot d)$ операций. Дополнительно — умножение на $V$: также $O(T^2 \cdot d)$.

Таким образом, сложность Transformer составляет $O(T^2 \cdot d)$, а RNN — $O(T \cdot d^2)$. В большинстве практических задач $T$ (длина последовательности) значительно меньше $d$ (размерности представления). Например, в машинном переводе типичные значения: $T \approx 30$–50, $d = 512$, так что $T^2 \cdot d \approx 2500 \cdot 512 \approx 1.3 \cdot 10^6$, в то время как $T \cdot d^2 \approx 50 \cdot 262144 \approx 13 \cdot 10^6$ — разница в 10 раз в пользу Transformer. При больших $T$ (например, длинные документы) ситуация меняется, но на практике для большинства задач длина последовательности ограничена бюджетом памяти и составляет несколько тысяч токенов, тогда как $d$ может быть 4096 и более.

Это подтверждается экспериментальными данными: Transformer обучается в 3–10 раз быстрее RNN-аналогов при сопоставимом качестве. В оригинальной статье авторы отмечают, что базовая модель обучается всего **12 часов** на 8 GPU P100, тогда как лучшие RNN-модели того времени требовали недель.

На диаграмме ниже показано сравнение вычислительной сложности для разных типов слоёв:

```mermaid
xychart-beta
    title "Зависимость времени выполнения от длины последовательности (условно)"
    x-axis "Длина последовательности T" 10 --> 1000
    y-axis "Время (условные единицы)" 0 --> 1000
    line "RNN (O(T·d²))" [1, 10, 100, 1000]
    line "Transformer (O(T²·d))" [1, 4, 100, 10000]
```

*Примечание: при малых T (до ∼100) Transformer быстрее; при очень длинных последовательностях RNN может оказаться эффективнее, но на практике используются оптимизации (Flash Attention, локальное внимание).*

Кроме того, Transformer полностью использует возможности GPU: операции с матрицами могут быть выполнены с высокой степенью параллелизма, достигая практически 100% загрузки вычислительных ядер. Это делает обучение больших моделей на кластерах GPU экономически эффективным.

---

### 2. Глобальный контекст: каждый токен видит все остальные

Второе критическое преимущество — возможность прямого взаимодействия между любыми двумя позициями последовательности с **постоянным числом операций** ($O(1)$), независимо от расстояния между ними. В RNN для связи первого и последнего токена необходимо пройти через все промежуточные шаги, что требует $O(T)$ операций и сталкивается с проблемой затухающих градиентов.

Transformer достигает этого благодаря механизму Scaled Dot-Product Attention:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.
$$

Матрица $QK^T$ размера $(T \times T)$ содержит скалярные произведения, которые являются мерой совместимости каждой пары токенов. После применения softmax получаются веса внимания, показывающие, насколько каждый токен (запрос) должен обращать внимание на каждый другой токен (ключ). Затем эти веса используются для взвешенного суммирования значений $V$.

Интерпретация этого механизма: каждый токен получает новое представление как **взвешенная сумма всех токенов последовательности**, где веса определяются их смысловой близостью. Таким образом, модель может напрямую использовать информацию из любого места последовательности, что особенно важно для задач, требующих учёта дальних зависимостей (например, разрешение анафор, извлечение отношений).

В статье авторы подчёркивают этот аспект, сравнивая максимальную длину пути для распространения сигнала:

> *«Самовнимание связывает все позиции с постоянным числом последовательно выполняемых операций, тогда как рекуррентный слой требует $O(n)$ последовательных операций»* (раздел 4, таблица 1).

Более того, авторы отмечают, что множественные головы внимания позволяют модели одновременно фокусироваться на различных типах зависимостей: синтаксических, семантических, локальных и глобальных. Это даёт Transformer глубокое понимание структуры текста.

---

### 3. Масштабируемость: законы, которые работают

Transformer оказался архитектурой, которая **масштабируется** — увеличение размера модели (числа параметров), объёма данных и вычислительных ресурсов приводит к предсказуемому улучшению качества. Это свойство было формализовано в работе **Kaplan et al. (2020)** «Scaling Laws for Neural Language Models» [2], которая установила степенные зависимости между качеством модели и этими тремя факторами:

$$
\mathcal{L} \propto \left(\frac{N}{N_0}\right)^{\alpha_N} \cdot \left(\frac{D}{D_0}\right)^{\alpha_D} \cdot \left(\frac{C}{C_0}\right)^{\alpha_C},
$$

где:

- $N$ — число параметров модели,
- $D$ — размер обучающего корпуса (в токенах),
- $C$ — вычислительные затраты (FLOPs),
- $\alpha_N, \alpha_D, \alpha_C$ — отрицательные показатели степени (обычно около $-0.05$ – $-0.1$).

Это означает, что для улучшения качества на 10% требуется примерно в 10 раз больше параметров или данных. Однако, в отличие от RNN, Transformer демонстрирует устойчивое масштабирование без насыщения вплоть до триллионов параметров (GPT-4, 1.8 трлн). Причины этого:

1. **Отсутствие рекуррентных связей** — градиенты не затухают и не взрываются, что позволяет обучать модели с сотнями слоёв.
2. **Эффективная параллелизация** — обучение можно распределить на тысячи GPU, используя такие методы, как модель параллелизма (model parallelism) и смешанная точность.
3. **Гибкость архитектуры** — возможно вводить разреженность (MoE) и другие оптимизации без изменения основного принципа.

Ниже представлена таблица, обобщающая законы масштабирования для Transformer:

| Параметр | Обозначение | Типичный показатель степени | Влияние на качество |
|----------|-------------|-----------------------------|----------------------|
| Число параметров | $N$ | $\alpha_N \approx -0.076$ | Увеличение в 10 раз → снижение потерь на ~16% |
| Размер данных | $D$ | $\alpha_D \approx -0.095$ | Увеличение в 10 раз → снижение потерь на ~20% |
| Вычисления | $C$ | $\alpha_C \approx -0.050$ | Увеличение в 10 раз → снижение потерь на ~11% |

*Источник: Kaplan et al., 2020.*

Эти законы стали руководством для индустрии: оптимальное соотношение — увеличивать все три фактора одновременно, что и привело к появлению моделей масштаба GPT-3, LLaMA-3 и других.

---

### 4. Универсальность: одна архитектура для всех модальностей

Четвёртое преимущество Transformer — его **универсальность**. Одна и та же архитектура, с минимальными модификациями, оказалась применимой к широкому спектру задач и типов данных.

#### 4.1. В обработке естественного языка (NLP)

Transformer породил два основных направления:

- **Encoder-Only** (например, BERT) — для задач понимания текста: классификация, извлечение информации, ответы на вопросы.
- **Decoder-Only** (например, GPT) — для генеративных задач: написание текста, кода, диалогов.
- **Encoder-Decoder** (оригинальный Transformer, T5) — для задач преобразования последовательностей: машинный перевод, суммаризация.

Все современные LLM — GPT-4, Claude, LLaMA, Qwen, Gemini — являются вариациями Transformer.

#### 4.2. В компьютерном зрении (CV)

В 2020 году был предложен **Vision Transformer (ViT)** [3], который применяет Transformer к изображениям, разбивая их на патчи и обрабатывая их как последовательность. ViT показал результаты, сопоставимые с лучшими свёрточными сетями (ResNet, EfficientNet) на ImageNet, а при масштабировании — превосходящие их.

#### 4.3. В обработке аудио

**Audio Spectrogram Transformer** [4] и **Whisper** от OpenAI используют Transformer для распознавания речи и классификации звуков. Спектрограмма преобразуется в последовательность, и модель обрабатывает её так же, как текст.

#### 4.4. В мультимодальных системах

Модели типа **CLIP**, **Flamingo**, **GPT-4o** объединяют текст, изображения, видео и аудио, используя Transformer как универсальный кодировщик, способный выравнивать представления разных модальностей в общем пространстве.

Авторы оригинальной статьи предвидели это расширение:

> *«Мы с воодушевлением смотрим в будущее моделей, основанных на внимании, и планируем применить их к другим задачам. Мы планируем расширить Transformer на задачи, включающие другие модальности ввода и вывода, помимо текста»*.

---

### Сводное сравнение

В таблице ниже обобщены ключевые различия между архитектурами:

| Критерий | RNN / LSTM | Transformer |
|----------|------------|-------------|
| Последовательная обработка | Да ($O(T)$ шагов) | Нет (все шаги параллельно) |
| Связь дальних позиций | $O(T)$ операций | $O(1)$ операций |
| Вычислительная сложность (на слой) | $O(T \cdot d^2)$ | $O(T^2 \cdot d)$ |
| Параллелизация | Слабая | Полная |
| Проблема затухающих градиентов | Присутствует | Отсутствует (прямые пути) |
| Применимость к разным модальностям | Текст, временные ряды | Текст, изображения, аудио, видео |

---

### Заключение

Преимущества Transformer — параллелизация, глобальный контекст, масштабируемость и универсальность — сделали его архитектурой, определившей развитие ИИ в последнее десятилетие. Отказ от рекуррентности позволил не только ускорить обучение, но и открыл возможности для создания моделей, которые сегодня составляют основу генеративного искусственного интеллекта. Понимание этих преимуществ необходимо для осознанного выбора архитектуры при решении прикладных задач и для дальнейшего развития методов глубокого обучения.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Kaplan, J., et al. (2020). *Scaling Laws for Neural Language Models*. arXiv:2001.08361.  
   🔗 [https://arxiv.org/abs/2001.08361](https://arxiv.org/abs/2001.08361)

3. Dosovitskiy, A., et al. (2020). *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. ICLR.  
   🔗 [https://arxiv.org/abs/2010.11929](https://arxiv.org/abs/2010.11929)

4. Gong, Y., et al. (2021). *AST: Audio Spectrogram Transformer*. Interspeech.  
   🔗 [https://arxiv.org/abs/2104.01778](https://arxiv.org/abs/2104.01778)

## Тема 1.4. Три типа архитектур на основе Transformer

Архитектура Transformer, предложенная в оригинальной статье [1], оказалась не просто удачным решением для машинного перевода — она породила целое семейство архитектур, каждая из которых адаптирована под определённый класс задач. В зависимости от того, какая часть исходной архитектуры используется — только энкодер, только декодер или оба компонента вместе, — модели приобретают различные свойства и области применения. В этом разделе мы рассмотрим три типа архитектур на основе Transformer: **Encoder-Only**, **Decoder-Only** и **Encoder-Decoder**.

---

### 1. Encoder-Only: архитектура для понимания

#### 1.1. Архитектура

Модели типа Encoder-Only используют только стек энкодеров Transformer. В оригинальной статье энкодер представляет собой последовательность $N$ идентичных слоёв (в оригинале $N=6$), каждый из которых содержит два подслоя:

1. **Multi-Head Self-Attention** — позволяет каждому токену взаимодействовать со всеми другими токенами в обе стороны (бидирекционально);
2. **Position-wise Feed-Forward Network** — применяется независимо к каждой позиции.

После каждого подслоя используются остаточные связи и нормализация. Ключевая особенность — **отсутствие маскировки**: каждый токен может обращаться к любому другому токену, включая те, что находятся справа от него.

#### 1.2. Обучение: Masked Language Model (MLM)

Encoder-Only модели обучаются с использованием **маскированного языкового моделирования** (Masked Language Model, MLM). Этот метод был предложен в работе BERT [2] и заключается в следующем:

1. Входная последовательность случайным образом маскируется: 15% токенов заменяются на специальный токен `[MASK]`;
2. Модель должна предсказать исходные токены на основе контекста — токенов слева и справа;
3. Обучается только на предсказании замаскированных токенов (кросс-энтропийная потеря).

Математически цель обучения можно записать как:

$$
\mathcal{L}_{\text{MLM}} = -\sum_{i \in \mathcal{M}} \log P(x_i \mid x_{\setminus i}),
$$

где $\mathcal{M}$ — множество замаскированных позиций, а $x_{\setminus i}$ — все остальные токены в последовательности.

Дополнительно в BERT используется задача **Next Sentence Prediction (NSP)**, которая обучает модель пониманию связи между предложениями.

#### 1.3. Задачи

Encoder-Only модели специализируются на задачах **понимания текста**, где требуется извлечь информацию из уже имеющегося текста, а не генерировать новый. Типичные применения:

- **Классификация текста** — определение тональности, тематики;
- **Named Entity Recognition (NER)** — извлечение именованных сущностей;
- **Question Answering (QA)** — поиск ответа в тексте;
- **Извлечение информации** — структурирование данных из неструктурированного текста;
- **Определение семантической близости** — сравнение смысла текстов.

#### 1.4. Примеры моделей

- **BERT** (Bidirectional Encoder Representations from Transformers) — первая и наиболее известная модель, Google, 2018;
- **RoBERTa** — улучшенная версия BERT от Facebook, 2019;
- **DistilBERT** — дистиллированная версия BERT для мобильных устройств;
- **ALBERT** — лёгкая версия BERT с разделением параметров;
- **ELECTRA** — модель с заменой токенов вместо маскирования.

#### 1.5. Преимущества и ограничения

**Преимущества:**
- **Полный двусторонний контекст** — каждый токен видит информацию как слева, так и справа, что критически важно для понимания языка;
- **Высокое качество** на задачах классификации и извлечения информации;
- **Эффективное использование** предобученных знаний.

**Ограничения:**
- **Не способна к генерации** связных текстов (нельзя использовать как чат-бота);
- Требует специального формата для каждой задачи (например, токен `[CLS]` для классификации);
- Ограничена задачами понимания, а не творческого синтеза.

На диаграмме ниже показана архитектура Encoder-Only:

```mermaid
flowchart TD
    subgraph Input["Входные данные"]
        S[Токены: x₁, x₂, ..., x_T]
    end

    subgraph Embedding["Слой эмбеддингов"]
        E[Эмбеддинги + Positional Encoding]
    end

    subgraph EncoderStack["Стек энкодеров (N×)"]
        direction TB
        EN1[Энкодер слой 1]
        EN2[Энкодер слой 2]
        ENDot[⋯]
        ENN[Энкодер слой N]
    end

    subgraph EncoderLayer["Слой энкодера"]
        MHA1[Multi-Head Self-Attention<br/><b>без маски</b>]
        AD1[Add & LayerNorm]
        FFN1[Feed-Forward Network]
        AD2[Add & LayerNorm]
    end

    subgraph Output["Выходной слой"]
        OUT[Выходные представления<br/>контекстуализированные]
        TASK[Задача: классификация, NER, QA]
    end

    S --> E --> EncoderStack
    E --> MHA1 --> AD1 --> FFN1 --> AD2
    AD2 --> EN2

    ENN --> OUT --> TASK

    style Input fill:#e3f2fd
    style Embedding fill:#e8f5e9
    style EncoderStack fill:#f3e5f5
    style Output fill:#e0f7fa
```

#### 1.6. Пример: вход и выход

```
Вход:  "Я [MASK] этот фильм."
Обучение: модель предсказывает "люблю" или "ненавижу"
Задача: тональность ("люблю") → положительная классификация
```

---

### 2. Decoder-Only: архитектура для генерации

#### 2.1. Архитектура

Модели типа Decoder-Only используют только стек декодеров Transformer. Декодер в оригинальной архитектуре содержит три подслоя:

1. **Masked Multi-Head Self-Attention** — позволяет каждому токену взаимодействовать только с предыдущими токенами (каузальная маска);
2. **Cross-Attention** — внимание к выходу энкодера (в Decoder-Only моделях этот слой обычно отсутствует, так как энкодера нет);
3. **Position-wise Feed-Forward Network**.

Ключевая особенность — **каузальная маска**, которая запрещает токену обращаться к будущим токенам (тем, что находятся правее). Это обеспечивает авторегрессивность: модель может генерировать текст слева направо, по одному токену за раз.

#### 2.2. Обучение: авторегрессивное языковое моделирование

Decoder-Only модели обучаются на задаче **авторегрессивного языкового моделирования** (Causal Language Modeling, CLM). Модель учится предсказывать следующий токен на основе предыдущих:

$$
P(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \dots, x_{t-1}),
$$

где $x_t$ — токен на позиции $t$. Функция потерь — кросс-энтропия между предсказанным распределением и истинным токеном:

$$
\mathcal{L}_{\text{CLM}} = -\sum_{t=1}^{T} \log P(x_t \mid x_{<t}).
$$

После обучения модель может генерировать текст, последовательно предсказывая следующий токен, используя методы декодирования (жадный поиск, beam search, температурная выборка).

#### 2.3. Задачи

Decoder-Only модели — это **генеративные модели**, способные создавать новый текст. Основные применения:

- **Генерация текста** — написание статей, стихов, историй;
- **Диалоговые системы** — чат-боты, виртуальные ассистенты;
- **Код-ассистенты** — написание и дополнение кода;
- **Машинный перевод** — генерация перевода;
- **Суммаризация** — создание кратких изложений;
- **Ответы на вопросы** — порождение ответа в свободной форме.

#### 2.4. Примеры моделей

- **GPT** (Generative Pre-trained Transformer) — семейство моделей OpenAI (GPT-1, GPT-2, GPT-3, GPT-4);
- **LLaMA** — открытые модели от Meta (LLaMA 1, 2, 3);
- **Qwen** — открытые модели от Alibaba;
- **Mistral** — открытые модели от Mistral AI;
- **Claude** — модели от Anthropic;
- **Gemini** — модели от Google (частично).

#### 2.5. Преимущества и ограничения

**Преимущества:**
- **Способность к генерации** — может создавать связные, осмысленные тексты;
- **Универсальность** — одна модель решает множество задач через промпт-инжиниринг;
- **Zero-shot и few-shot обучение** — способность решать новые задачи по инструкции;
- **Масштабируемость** — хорошо масштабируется с ростом параметров и данных.

**Ограничения:**
- **Односторонний контекст** — токен не видит того, что будет справа, что может быть неоптимально для задач понимания;
- **Тенденция к галлюцинациям** — может выдумывать факты при недостатке информации;
- **Высокие вычислительные затраты** на инференс (генерация требует последовательного вычисления токенов).

На диаграмме ниже показана архитектура Decoder-Only:

```mermaid
flowchart TD
    subgraph Input["Входные данные"]
        S[Токены: x₁, x₂, ..., x_T]
    end

    subgraph Embedding["Слой эмбеддингов"]
        E[Эмбеддинги + Positional Encoding]
    end

    subgraph DecoderStack["Стек декодеров (N×)"]
        direction TB
        DN1[Декодер слой 1]
        DN2[Декодер слой 2]
        DNDot[⋯]
        DNN[Декодер слой N]
    end

    subgraph DecoderLayer["Слой декодера"]
        MMHA[Masked Multi-Head Self-Attention<br/><b>с каузальной маской</b>]
        AD3[Add & LayerNorm]
        FFN2[Feed-Forward Network]
        AD5[Add & LayerNorm]
    end

    subgraph Output["Выходной слой"]
        LC[Линейный слой + Softmax]
        GEN[Генерация следующего токена]
    end

    S --> E --> DecoderStack
    E --> MMHA --> AD3 --> FFN2 --> AD5
    AD5 --> DN2

    DNN --> LC --> GEN --> O[Следующий токен]

    style Input fill:#e3f2fd
    style Embedding fill:#e8f5e9
    style DecoderStack fill:#fce4ec
    style Output fill:#e0f7fa
```

#### 2.6. Пример: вход и выход

```
Вход:  "Я люблю этот фильм, потому что он"
Генерация: "очень интересный и захватывающий."
```

---

### 3. Encoder-Decoder: архитектура для преобразования

#### 3.1. Архитектура

Модели типа Encoder-Decoder используют полную архитектуру Transformer, включающую как стек энкодеров, так и стек декодеров. Эта архитектура соответствует оригинальному Transformer и включает:

1. **Энкодер** — обрабатывает входную последовательность, создавая контекстуализированные представления;
2. **Декодер** — генерирует выходную последовательность, используя:
   - Masked Self-Attention (для авторегрессии);
   - Cross-Attention (внимание к выходу энкодера);
   - Feed-Forward Network.

Cross-Attention является ключевым компонентом: запросы $Q$ приходят из декодера, а ключи $K$ и значения $V$ — из выхода энкодера. Это позволяет декодеру «смотреть» на всю входную последовательность при генерации каждого токена.

#### 3.2. Обучение: sequence-to-sequence

Encoder-Decoder модели обучаются на задачах **sequence-to-sequence** (последовательность-в-последовательность), где входная и выходная последовательности могут иметь разную длину. Классический подход — **авторегрессивное обучение с учителем**:

1. На вход подаётся последовательность $X$ (например, предложение на английском);
2. Модель генерирует последовательность $Y$ (например, перевод на немецкий);
3. Обучение происходит через минимизацию кросс-энтропийной потери между сгенерированными и истинными токенами:

$$
\mathcal{L}_{\text{seq2seq}} = -\sum_{t=1}^{T_y} \log P(y_t \mid y_{<t}, X).
$$

#### 3.3. Задачи

Encoder-Decoder модели специализируются на задачах **преобразования** одной последовательности в другую:

- **Машинный перевод** — перевод текста с одного языка на другой;
- **Суммаризация** — сокращение длинного текста до краткого изложения;
- **Перефразирование** — пересказ текста другими словами;
- **Ответы на вопросы по тексту** (generative QA);
- **Генерация по структуре** — текст по таблице, диаграмме.

#### 3.4. Примеры моделей

- **Оригинальный Transformer** — архитектура для машинного перевода;
- **T5** (Text-to-Text Transfer Transformer) — унифицированная модель от Google, все задачи формулируются как «текст в текст»;
- **BART** — модель от Facebook, сочетающая BERT-подобное обучение и GPT-подобную генерацию;
- **Pegasus** — модель для суммаризации от Google;
- **mT5** — мультиязычная версия T5.

#### 3.5. Преимущества и ограничения

**Преимущества:**
- **Оптимальна для преобразований** — где структура входа и выхода различна;
- **Гибкость** — можно использовать для любых задач «текст-в-текст»;
- **Качество** — часто превосходит Decoder-Only на структурированных задачах.

**Ограничения:**
- **Больше параметров** — содержит и энкодер, и декодер;
- **Дороже в обучении** и инференсе;
- **Менее универсальна** — не подходит для открытых диалогов, где нет явного входа.

На диаграмме ниже показана полная архитектура Encoder-Decoder:

```mermaid
flowchart TD
    subgraph Input["Входные данные"]
        S[Входная последовательность: x₁, x₂, ..., x_T]
    end

    subgraph Encoder["Энкодер (N×)"]
        ENC[Стек энкодеров<br/>с Self-Attention]
    end

    subgraph Decoder["Декодер (N×)"]
        DEC[Стек декодеров<br/>с Masked Self-Attention<br/>и Cross-Attention]
    end

    subgraph Output["Выходные данные"]
        OUT[Генерация y₁, y₂, ..., y_M]
    end

    S --> Encoder -->|K, V| Decoder
    Decoder --> Output

    style Input fill:#e3f2fd
    style Encoder fill:#f3e5f5
    style Decoder fill:#fce4ec
    style Output fill:#e0f7fa
```

#### 3.6. Пример: вход и выход

```
Вход:  "The cat sat on the mat."
Выход: "Кот сидел на коврике."
```

---

### 4. Сравнительный анализ

#### 4.1. Сводная схема всех трёх архитектур

На диаграмме ниже представлены все три типа архитектур вместе для наглядного сравнения:

```mermaid
flowchart LR
    subgraph EO["Encoder-Only (BERT)"]
        direction LR
        E1[Энкодер] --> E2[Энкодер] --> E3[...] --> E4[Энкодер] --> EOut[Выход<br/>Понимание]
    end

    subgraph DO["Decoder-Only (GPT)"]
        direction LR
        D1[Декодер] --> D2[Декодер] --> D3[...] --> D4[Декодер] --> DOut[Выход<br/>Генерация]
    end

    subgraph ED["Encoder-Decoder (T5)"]
        direction LR
        En1[Энкодер] --> En2[Энкодер] --> En3[...] --> En4[Энкодер] -->|K, V| De1[Декодер] --> De2[Декодер] --> De3[...] --> De4[Декодер] --> EDOut[Выход<br/>Преобразование]
    end

    style EO fill:#e3f2fd
    style DO fill:#fce4ec
    style ED fill:#f3e5f5
```

#### 4.2. Сравнительная таблица

| Критерий | Encoder-Only (BERT) | Decoder-Only (GPT) | Encoder-Decoder (T5) |
|----------|---------------------|-------------------|----------------------|
| **Архитектура** | Только энкодер | Только декодер | Энкодер + декодер |
| **Обучение** | MLM (маскирование) | Авторегрессия (next token) | Seq2seq с учителем |
| **Контекст** | Двусторонний | Односторонний (causal) | Двусторонний (энкодер) + односторонний (декодер) |
| **Основные задачи** | Понимание, классификация, NER | Генерация, диалоги, кодинг | Перевод, суммаризация |
| **Примеры моделей** | BERT, RoBERTa | GPT, LLaMA, Qwen | T5, BART |
| **Параллелизация** | Полная | Частичная (инференс последовательный) | Полная (энкодер) + частичная (декодер) |
| **Вычислительная сложность** | $O(T^2 \cdot d)$ | $O(T^2 \cdot d)$ (обучение)<br/>$O(T \cdot d^2)$ (инференс) | $O(T_x^2 \cdot d + T_y^2 \cdot d)$ |
| **Способность к генерации** | Нет | Да (высокая) | Да |
| **Способность к пониманию** | Да (высокая) | Ограниченная | Да (высокая) |
| **Размер параметров** | Средний | Большой | Самый большой |

#### 4.3. Когда какую архитектуру выбирать

Выбор архитектуры определяется характером задачи:

| Тип задачи | Рекомендуемая архитектура | Причина |
|------------|---------------------------|---------|
| Классификация текста | Encoder-Only | Двусторонний контекст даёт лучшее понимание |
| Извлечение информации (NER, QA) | Encoder-Only | Точное понимание текста критично |
| Генерация текста, диалоги | Decoder-Only | Оптимальна для авторегрессивной генерации |
| Код-ассистент | Decoder-Only | Генерация требует одностороннего потока |
| Машинный перевод | Encoder-Decoder | Требуется понимание входа и генерация выхода |
| Суммаризация | Encoder-Decoder | Комбинация понимания и сжатия |
| Ответы на вопросы (extractive) | Encoder-Only | Поиск ответа в тексте |
| Ответы на вопросы (generative) | Decoder-Only или Encoder-Decoder | Зависит от сложности входа |

---

### Заключение

Три типа архитектур на основе Transformer — Encoder-Only, Decoder-Only и Encoder-Decoder — представляют собой различные варианты использования механизма самовнимания для решения разных классов задач. Понимание различий между ними позволяет осознанно выбирать архитектуру для конкретной задачи, оценивать её возможности и ограничения, а также правильно интерпретировать поведение моделей. Современные исследования показывают, что Decoder-Only архитектуры становятся всё более универсальными, однако Encoder-Only и Encoder-Decoder по-прежнему сохраняют свои ниши, особенно в задачах, требующих глубокого понимания или структурированного преобразования.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

3. Radford, A., et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.  
   🔗 [https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

4. Raffel, C., et al. (2019). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*. JMLR.  
   🔗 [https://arxiv.org/abs/1910.10683](https://arxiv.org/abs/1910.10683)

5. Lewis, M., et al. (2019). *BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension*. ACL.  
   🔗 [https://arxiv.org/abs/1910.13461](https://arxiv.org/abs/1910.13461)

## Тема 2.1. Структура Encoder-Decoder: фундаментальная архитектура Transformer

Архитектура Transformer в своей оригинальной конфигурации [1] представляет собой **симметричную структуру «энкодер-декодер»** (encoder-decoder), предназначенную для задач преобразования последовательностей. В данном разделе мы дадим исчерпывающее математическое описание каждого компонента — от входного слоя до генерации выходного токена, — чтобы сформировать полное понимание потоков данных и преобразований.

---

### 1. Общая схема и поток данных

Пусть дана входная последовательность $\mathbf{x} = (x_1, \dots, x_T)$ длины $T$ (например, слова на исходном языке) и целевая последовательность $\mathbf{y} = (y_1, \dots, y_M)$ длины $M$ (например, перевод). Архитектура Encoder-Decoder преобразует $\mathbf{x}$ в $\mathbf{y}$ через следующие этапы:

1. **Энкодер** преобразует $\mathbf{x}$ в матрицу контекстных представлений $Z \in \mathbb{R}^{T \times d_{\text{model}}}$.
2. **Декодер** авторегрессивно генерирует $\mathbf{y}$, на каждом шаге $t$ используя $Z$ и уже сгенерированные токены $y_{<t}$.

Математически это записывается как:

$$
Z = \text{Encoder}(\mathbf{x}), \qquad
P(y_t \mid y_{<t}, \mathbf{x}) = \text{Decoder}(y_{<t}, Z).
$$

Полный поток данных показан на диаграмме:

```mermaid
flowchart TD
    subgraph Input["Входной текст"]
        X[("x₁, x₂, ..., x_T")]
    end

    subgraph Encoder["Энкодер"]
        direction TB
        ENC_EMB[Эмбеддинги + PE]
        ENC_LAYERS[Стек энкодеров N×]
        Z[("Z ∈ ℝ^{T×d_model}")]
    end

    subgraph Decoder["Декодер"]
        direction TB
        DEC_EMB[Эмбеддинги + PE для выходных токенов]
        DEC_LAYERS[Стек декодеров N×]
        H[("Скрытые состояния")]
    end

    subgraph Output["Генерация"]
        PROJ[Линейный слой + Softmax]
        Y[("y₁, y₂, ..., y_M")]
    end

    X --> ENC_EMB --> ENC_LAYERS --> Z
    Z -->|"K, V через Cross-Attention"| DEC_LAYERS
    Y -->|"авторегрессивно"| DEC_EMB --> DEC_LAYERS
    DEC_LAYERS --> H --> PROJ --> Y

    style Input fill:#e3f2fd
    style Encoder fill:#f3e5f5
    style Decoder fill:#fce4ec
    style Output fill:#e0f7fa
```

---

### 2. Энкодер: математика преобразования входа в контекст

#### 2.1. Входной слой: эмбеддинги и позиционное кодирование

Каждый токен $x_i$ отображается в вектор размерности $d_{\text{model}}$ с помощью обучаемой матрицы эмбеддингов $E \in \mathbb{R}^{V \times d_{\text{model}}}$, где $V$ — размер словаря:

$$
\mathbf{e}_i = E[x_i] \in \mathbb{R}^{d_{\text{model}}}.
$$

Затем добавляется позиционное кодирование $\mathbf{p}_i \in \mathbb{R}^{d_{\text{model}}}$, которое в оригинале определяется синусоидальными функциями:

$$
\begin{aligned}
\mathbf{p}_{i, 2j} &= \sin\left(\frac{i}{10000^{2j/d_{\text{model}}}}\right), \\
\mathbf{p}_{i, 2j+1} &= \cos\left(\frac{i}{10000^{2j/d_{\text{model}}}}\right),
\end{aligned}
$$

где $j$ — индекс измерения. В результате получается входная матрица энкодера:

$$
\mathbf{X} = [\mathbf{e}_1 + \mathbf{p}_1, \dots, \mathbf{e}_T + \mathbf{p}_T] \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

#### 2.2. Стек слоёв энкодера

Энкодер состоит из $N$ идентичных слоёв (в оригинале $N=6$). Каждый слой содержит два подслоя:

1. **Multi-Head Self-Attention** (без маскировки).
2. **Position-wise Feed-Forward Network (FFN)**.

Вокруг каждого подслоя применяется **остаточная связь** и **нормализация** (в оригинале — пост-нормализация, но современные реализации часто используют пре-нормализацию). Для определённости опишем пост-нормализацию, как в оригинале:

$$
\begin{aligned}
\mathbf{X}^{(1)} &= \text{LayerNorm}\big(\mathbf{X} + \text{MHA}(\mathbf{X})\big), \\
\mathbf{X}^{(2)} &= \text{LayerNorm}\big(\mathbf{X}^{(1)} + \text{FFN}(\mathbf{X}^{(1)})\big),
\end{aligned}
$$

где $\mathbf{X}$ — вход слоя, $\mathbf{X}^{(2)}$ — выход слоя.

##### 2.2.1. Multi-Head Self-Attention (MHA)

Сначала вычисляются запросы, ключи и значения для всех позиций:

$$
\mathbf{Q} = \mathbf{X} W^Q, \quad \mathbf{K} = \mathbf{X} W^K, \quad \mathbf{V} = \mathbf{X} W^V,
$$

где $W^Q, W^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$, $W^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$, причём $d_k = d_v = d_{\text{model}} / h$ (в оригинале $h=8$).

Затем вычисляется $h$ голов внимания:

$$
\text{head}_i = \text{Attention}\left(\mathbf{Q} W_i^Q,\; \mathbf{K} W_i^K,\; \mathbf{V} W_i^V\right),
$$

где

$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}.
$$

Результаты голов конкатенируются и проецируются:

$$
\text{MHA}(\mathbf{X}) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) \, W^O,
$$

где $W^O \in \mathbb{R}^{h d_v \times d_{\text{model}}}$.

##### 2.2.2. Position-wise Feed-Forward Network (FFN)

FFN применяется к каждой позиции независимо:

$$
\text{FFN}(\mathbf{x}) = \max(0, \mathbf{x} W_1 + b_1) W_2 + b_2,
$$

где $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$, $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$, а $d_{\text{ff}} = 2048$ в оригинале.

##### 2.2.3. Layer Normalization

Нормализация выполняется по измерению признаков для каждой позиции отдельно:

$$
\text{LayerNorm}(\mathbf{x}) = \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta,
$$

где $\mu$ и $\sigma^2$ — среднее и дисперсия по $d_{\text{model}}$, $\gamma, \beta$ — обучаемые параметры.

#### 2.3. Выход энкодера

После прохождения всех $N$ слоёв получается матрица контекстных представлений:

$$
\mathbf{Z} = \text{Encoder}(\mathbf{X}) \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

Эта матрица содержит для каждого входного токена вектор, обогащённый информацией о всей последовательности.

---

### 3. Декодер: авторегрессивная генерация

Декодер генерирует выходную последовательность по одному токену за раз. На шаге $t$ он получает:

- матрицу $\mathbf{Z}$ из энкодера,
- уже сгенерированные токены $y_1, \dots, y_{t-1}$ (для $t=1$ — только специальный токен начала).

#### 3.1. Входной слой декодера

Аналогично энкодеру, каждый входной токен $y_i$ преобразуется в эмбеддинг и суммируется с позиционным кодированием (позиции соответствуют выходной последовательности). На шаге $t$ декодер использует матрицу $\mathbf{Y}_{<t} \in \mathbb{R}^{(t-1) \times d_{\text{model}}}$.

#### 3.2. Стек слоёв декодера

Каждый слой декодера содержит **три** подслоя:

1. **Masked Multi-Head Self-Attention** — позволяет каждому токену обращаться только к предыдущим токенам.
2. **Cross-Attention** (Encoder-Decoder Attention) — позволяет декодеру обращаться к $\mathbf{Z}$.
3. **Position-wise FFN**.

##### 3.2.1. Masked Self-Attention

Вычисления аналогичны обычному Self-Attention, но перед softmax в матрицу $\mathbf{Q}\mathbf{K}^T$ добавляется маска, которая обнуляет (устанавливает в $-\infty$) все позиции, соответствующие будущим токенам:

$$
\text{MaskedAttn}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right) \mathbf{V},
$$

где $\mathbf{M}_{ij} = 0$ при $j \le i$ и $-\infty$ при $j > i$.

##### 3.2.2. Cross-Attention

Здесь запросы $\mathbf{Q}$ берутся из предыдущего подслоя декодера, а ключи и значения — из выхода энкодера $\mathbf{Z}$:

$$
\mathbf{Q} = \mathbf{H} W^Q, \quad \mathbf{K} = \mathbf{Z} W^K, \quad \mathbf{V} = \mathbf{Z} W^V,
$$

и применяется обычное внимание без маски.

#### 3.3. Выходной слой

После стека декодеров получается скрытое состояние $\mathbf{H} \in \mathbb{R}^{M \times d_{\text{model}}}$. Затем применяется линейный слой и softmax для получения распределения вероятностей следующего токена:

$$
P(y_t \mid y_{<t}, \mathbf{x}) = \text{softmax}(\mathbf{H}_{t-1} W_{\text{out}} + b_{\text{out}}),
$$

где $\mathbf{H}_{t-1}$ — представление последнего сгенерированного токена (или начального токена) размерности $d_{\text{model}}$.

---

### 4. Детали взаимодействия: Cross-Attention и передача информации

Cross-Attention является ключевым мостом между энкодером и декодером. Он позволяет декодеру на каждом шаге динамически извлекать информацию из всей входной последовательности. Математически, для каждого слоя декодера:

$$
\mathbf{Q} = \text{LayerNorm}(\mathbf{H}_{\text{prev}} + \text{MaskedSelfAttn}(\mathbf{H}_{\text{prev}})),
$$
$$
\mathbf{K} = \mathbf{Z}, \quad \mathbf{V} = \mathbf{Z},
$$
$$
\mathbf{H}_{\text{new}} = \text{LayerNorm}(\mathbf{Q} + \text{CrossAttn}(\mathbf{Q}, \mathbf{K}, \mathbf{V})).
$$

Это гарантирует, что на каждом шаге генерации модель имеет доступ ко всему контексту входа.

#### Схема размерностей на каждом этапе

```mermaid
flowchart LR
    A["Вход: T×d_model"] --> B["Энкодер: T×d_model"]
    B --> C["Z: T×d_model"]
    C --> D["Cross-Attention<br/>Q:1×d_k, K:T×d_k, V:T×d_v"]
    D --> E["Выход: 1×d_v"]
    E --> F["Следующий токен"]
    
    style A fill:#e3f2fd
    style B fill:#f3e5f5
    style C fill:#f3e5f5
    style D fill:#fff3e0
    style E fill:#fce4ec
    style F fill:#e0f7fa
```

---

### 5. Полный пример: машинный перевод (с пошаговой математикой)

Рассмотрим перевод предложения **"I love reading books"** на русский.

#### Шаг 1: Токенизация и эмбеддинги
Токены: `["I", "love", "reading", "books"]` → эмбеддинги + PE → $\mathbf{X} \in \mathbb{R}^{4 \times 512}$.

#### Шаг 2: Энкодер
Через $N$ слоёв получаем $\mathbf{Z} \in \mathbb{R}^{4 \times 512}$. Каждый вектор $\mathbf{z}_i$ содержит контекст всего предложения.

#### Шаг 3: Генерация декодером

| Шаг $t$ | Вход декодера ($y_{<t}$) | Действие | Выходной токен |
|---------|--------------------------|----------|----------------|
| 1 | `[START]` | Cross-Attention к $\mathbf{Z}$ -> предсказание | `"Я"` |
| 2 | `[START]`, `"Я"` | Снова Cross-Attention, учитывая `"Я"` | `"люблю"` |
| 3 | ... `"люблю"` | ... | `"читать"` |
| 4 | ... `"читать"` | ... | `"книги"` |
| 5 | ... `"книги"` | ... | `[END]` |

На каждом шаге вычисляется:

$$
\mathbf{h}_t = \text{DecoderLayer}(\mathbf{h}_{t-1}, \mathbf{Z}), \quad
P(y_t) = \text{softmax}(\mathbf{h}_t W_{\text{out}}).
$$

#### Визуализация процесса

```mermaid
flowchart TD
    A["Вход: I love reading books"] --> B["Энкодер: Z"]
    B --> C1["Шаг1: [START] → Я"]
    B --> C2["Шаг2: [START] Я → люблю"]
    B --> C3["Шаг3: [START] Я люблю → читать"]
    B --> C4["Шаг4: [START] Я люблю читать → книги"]
    B --> C5["Шаг5: ... → [END]"]
    C1 --> D["Выход: Я люблю читать книги"]
    C2 --> D
    C3 --> D
    C4 --> D
    C5 --> D
```

---

### 6. Обучение и функция потерь

Обучение происходит с учителем на парах (вход, эталонный выход). Для каждого шага генерации вычисляется кросс-энтропийная потеря между предсказанным распределением и истинным токеном:

$$
\mathcal{L} = -\sum_{t=1}^{M} \log P(y_t^* \mid y_{<t}, \mathbf{x}),
$$

где $y_t^*$ — эталонный токен. Градиенты распространяются через всю архитектуру (энкодер и декодер) с помощью обратного распространения ошибки.

---

### 7. Роли энкодера и декодера (таблица)

| Аспект | Энкодер | Декодер |
|--------|---------|---------|
| **Задача** | Понимание входа, создание контекста | Генерация выхода |
| **Вход** | Токены исходного текста | Токены выхода (предыдущие) + представления энкодера |
| **Выход** | Матрица контекстных представлений $Z$ | Распределение вероятностей следующего токена |
| **Тип внимания** | Self-Attention (двусторонний) | Masked Self-Attention + Cross-Attention |
| **Параллелизм** | Полный (все позиции одновременно) | Частичный (инференс последовательный) |
| **Маска** | Отсутствует | Присутствует (для авторегрессии) |

---

### 8. Заключение

В этом разделе мы рассмотрели полную математическую модель архитектуры Encoder-Decoder Transformer, от входных эмбеддингов до генерации выходных токенов. Каждый компонент — позиционное кодирование, многоголовое внимание, остаточные связи, нормализация, кросс-внимание — был описан с использованием явных формул и размерностей. Эта архитектура, сохраняя свою актуальность в задачах перевода и суммаризации, заложила основу для всех последующих модификаций Transformer.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 2.2. Стек кодировщиков и декодировщиков

В предыдущем разделе мы рассмотрели общую архитектуру Encoder-Decoder и взаимодействие между её компонентами. Однако реальная мощь Transformer раскрывается в повторении идентичных слоёв — образовании **стека** (stack). Именно глубина стека позволяет модели выделять иерархические паттерны, переходить от поверхностных лексических связей к глубоким семантическим отношениям. В этом разделе мы детально разберём структуру отдельного слоя энкодера и декодера, их композицию в стек и эволюцию подходов к нормализации.

---

### 1. Структура стека: от слоя к глубине

#### 1.1. N идентичных слоёв

В оригинальной статье [1] Transformer состоит из стека **N = 6** идентичных слоёв как в энкодере, так и в декодере. Каждый слой имеет одинаковую архитектуру, но **не разделяет веса** — параметры каждого слоя обучаются независимо.

Почему слои не разделяют веса? В рекуррентных сетях (RNN) один и тот же слой применяется к каждому временному шагу (разделение весов по времени), что является следствием их рекуррентной природы. В Transformer, напротив, слои являются отдельными вычислительными блоками, и каждый из них может специализироваться на своём уровне абстракции. Исследования показывают, что нижние слои чаще фокусируются на локальных синтаксических связях (зависимости между соседними словами), средние — на семантических ролях, а верхние — на глобальном смысле и дискурсе [2]. Разделение весов позволило бы всем слоям выполнять одну и ту же функцию, что существенно ограничило бы выразительность модели.

#### 1.2. Как с каждым слоем растёт "глубина понимания"

Каждый слой энкодера преобразует входные представления, добавляя к ним всё более абстрактную контекстную информацию. На выходе первого слоя векторы токенов учитывают ближайшее окружение. После прохождения через несколько слоёв каждый вектор содержит информацию о всей последовательности, но на разных уровнях обобщения. Это аналогично тому, как в свёрточных сетях ранние слои выделяют края и текстуры, а глубокие — целые объекты.

Визуализация внимания в разных слоях Transformer подтверждает эту иерархию: ранние слои часто показывают локальное внимание (соседние слова), а поздние — глобальное (дальние зависимости и связи между предложениями) [3].

---

### 2. Слой энкодера: архитектура и поток данных

Каждый слой энкодера (рисунок 1) состоит из двух основных подслоёв, вокруг каждого из которых применяются остаточная связь и нормализация.

```mermaid
flowchart TD
    subgraph EncoderLayer["Слой энкодера"]
        direction TB
        Input[("Вход: X ∈ ℝ^{T×d_model}")]
        
        MHA["Подслой 1: Multi-Head Self-Attention<br/>Q, K, V из X (без маски)"]
        AD1["Add & LayerNorm"]
        FFN["Подслой 2: Position-wise FFN<br/>ReLU(W₁x + b₁)W₂ + b₂"]
        AD2["Add & LayerNorm"]
        
        Output[("Выход: X' ∈ ℝ^{T×d_model}")]
    end
    
    Input --> MHA
    MHA -->|" + X"| AD1
    AD1 --> FFN
    FFN -->|" + AD1"| AD2
    AD2 --> Output
    
    style Input fill:#e3f2fd
    style MHA fill:#f3e5f5
    style AD1 fill:#fff3e0
    style FFN fill:#f3e5f5
    style AD2 fill:#fff3e0
    style Output fill:#e0f7fa
```

#### 2.1. Подслой 1: Multi-Head Self-Attention

В этом подслое каждый токен взаимодействует со всеми другими токенами в последовательности. Поскольку маска отсутствует, внимание является **двусторонним** — токен может использовать информацию как слева, так и справа от себя.

Математически:

$$
\text{head}_i = \text{Attention}\left(\mathbf{X} W_i^Q,\; \mathbf{X} W_i^K,\; \mathbf{X} W_i^V\right),
$$

$$
\text{MHA}(\mathbf{X}) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O.
$$

Размерности остаются неизменными: вход $\mathbf{X} \in \mathbb{R}^{T \times d_{\text{model}}}$, выход также $\in \mathbb{R}^{T \times d_{\text{model}}}$.

#### 2.2. Подслой 2: Position-wise Feed-Forward Network

FFN применяется к каждой позиции независимо (position-wise) и состоит из двух линейных преобразований с функцией активации ReLU (в оригинале):

$$
\text{FFN}(\mathbf{x}) = \max(0, \mathbf{x} W_1 + b_1) W_2 + b_2,
$$

где $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$, $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$, причём $d_{\text{ff}} = 2048$ в оригинале (в 4 раза больше $d_{\text{model}} = 512$).

#### 2.3. Остаточные связи и нормализация

В оригинальной статье используется **пост-нормализация** (post-norm):

$$
\mathbf{X}^{(1)} = \text{LayerNorm}\left(\mathbf{X} + \text{MHA}(\mathbf{X})\right),
$$

$$
\mathbf{X}^{(2)} = \text{LayerNorm}\left(\mathbf{X}^{(1)} + \text{FFN}(\mathbf{X}^{(1)})\right).
$$

Остаточная связь $\mathbf{X} + \text{Sublayer}(\mathbf{X})$ позволяет градиентам свободно проходить через сеть, предотвращая проблему исчезающих градиентов. Нормализация стабилизирует распределение активаций.

---

### 3. Слой декодера: архитектура и поток данных

Декодер сложнее энкодера: он содержит **три** подслоя вместо двух. Это обусловлено необходимостью как авторегрессивной генерации, так и учёта выходных данных энкодера.

```mermaid
flowchart TD
    subgraph DecoderLayer["Слой декодера"]
        direction TB
        Input[("Вход: Y ∈ ℝ^{M×d_model}")]
        Z[("Выход энкодера: Z ∈ ℝ^{T×d_model}")]
        
        MMHA["Подслой 1: Masked Multi-Head Self-Attention<br/>Q, K, V из Y (с каузальной маской)"]
        AD1["Add & LayerNorm"]
        CA["Подслой 2: Cross-Attention<br/>Q из Y, K, V из Z"]
        AD2["Add & LayerNorm"]
        FFN["Подслой 3: Position-wise FFN"]
        AD3["Add & LayerNorm"]
        
        Output[("Выход: Y' ∈ ℝ^{M×d_model}")]
    end
    
    Input --> MMHA
    MMHA -->|" + Y"| AD1
    AD1 --> CA
    Z --> CA
    CA -->|" + AD1"| AD2
    AD2 --> FFN
    FFN -->|" + AD2"| AD3
    AD3 --> Output
    
    style Input fill:#e3f2fd
    style Z fill:#f3e5f5
    style MMHA fill:#fce4ec
    style AD1 fill:#fff3e0
    style CA fill:#ffcdd2
    style AD2 fill:#fff3e0
    style FFN fill:#fce4ec
    style AD3 fill:#fff3e0
    style Output fill:#e0f7fa
```

#### 3.1. Подслой 1: Masked Multi-Head Self-Attention

Этот подслой аналогичен Self-Attention в энкодере, но с **каузальной маской**, которая запрещает токену обращаться к будущим токенам.

Маска $\mathbf{M} \in \mathbb{R}^{M \times M}$ имеет вид:

$$
\mathbf{M}_{ij} =
\begin{cases}
0, & i \ge j \quad (\text{разрешено обращаться к прошлым и текущему}), \\
-\infty, & i < j \quad (\text{запрещено обращаться к будущим}).
\end{cases}
$$

Тогда:

$$
\text{MaskedAttn}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right) \mathbf{V}.
$$

#### 3.2. Подслой 2: Cross-Attention (Encoder-Decoder Attention)

Это ключевой мост между энкодером и декодером. Здесь:

- **Запросы $\mathbf{Q}$** приходят из предыдущего подслоя декодера (после Masked Self-Attention);
- **Ключи $\mathbf{K}$** и **Значения $\mathbf{V}$** приходят из выхода стека энкодеров $\mathbf{Z}$.

Это позволяет декодеру динамически извлекать информацию из входного текста на каждом шаге генерации.

$$
\text{CrossAttn}(\mathbf{Q}, \mathbf{Z}) = \text{Attention}\left(\mathbf{Q}, \mathbf{Z}W^K, \mathbf{Z}W^V\right).
$$

#### 3.3. Подслой 3: Position-wise Feed-Forward Network

Аналогичен FFN в энкодере и применяется независимо к каждой позиции.

#### 3.4. Нормализация в декодере

Аналогично энкодеру, в оригинале используется пост-нормализация:

$$
\mathbf{Y}^{(1)} = \text{LayerNorm}\left(\mathbf{Y} + \text{MaskedAttn}(\mathbf{Y})\right),
$$

$$
\mathbf{Y}^{(2)} = \text{LayerNorm}\left(\mathbf{Y}^{(1)} + \text{CrossAttn}(\mathbf{Y}^{(1)}, \mathbf{Z})\right),
$$

$$
\mathbf{Y}^{(3)} = \text{LayerNorm}\left(\mathbf{Y}^{(2)} + \text{FFN}(\mathbf{Y}^{(2)})\right).
$$

---

### 4. Стек из N слоёв

Каждый слой энкодера или декодера получает на вход выход предыдущего слоя. Таким образом, стек можно представить как композицию функций:

$$
\text{EncoderStack}(\mathbf{X}) = f_N \circ f_{N-1} \circ \dots \circ f_1(\mathbf{X}),
$$

где $f_i$ — $i$-й слой энкодера. Эта композиция позволяет модели строить всё более абстрактные представления.

```mermaid
flowchart TD
    subgraph EncoderStack["Стек энкодеров (N×)"]
        direction TB
        Input[("Вход: X₀ ∈ ℝ^{T×d_model}")]
        L1["Слой 1"]
        L2["Слой 2"]
        L3["..."]
        LN["Слой N"]
        Output[("Выход: Z = X_N ∈ ℝ^{T×d_model}")]
    end
    
    Input --> L1
    L1 --> L2
    L2 --> L3
    L3 --> LN
    LN --> Output
    
    style Input fill:#e3f2fd
    style L1 fill:#f3e5f5
    style L2 fill:#f3e5f5
    style L3 fill:#f3e5f5
    style LN fill:#f3e5f5
    style Output fill:#e0f7fa
```

---

### 5. Post-Norm vs Pre-Norm: эволюция нормализации

Оригинальный Transformer использовал **пост-нормализацию** (post-norm). Однако при масштабировании моделей (увеличении числа слоёв и параметров) этот подход стал приводить к нестабильности обучения. Современные модели (GPT, LLaMA, Qwen) используют **пре-нормализацию** (pre-norm).

#### 5.1. Сравнение подходов

| Аспект | Post-Norm (оригинал) | Pre-Norm (современный) |
|--------|----------------------|------------------------|
| **Формула** | $\text{LayerNorm}(x + \text{Sublayer}(x))$ | $x + \text{Sublayer}(\text{LayerNorm}(x))$ |
| **Порядок** | Сложение → Нормализация | Нормализация → Сложение |
| **Градиенты** | Затухают в глубоких сетях | Проходят напрямую через остаточные связи |
| **Стабильность** | Требует осторожной инициализации и LR | Устойчива к большим LR и глубоким сетям |
| **Использование** | Оригинальный Transformer | GPT, LLaMA, Mistral, Qwen |

#### 5.2. Математический анализ

В **Post-Norm** выход слоя имеет вид:

$$
\mathbf{x}_{out} = \text{LayerNorm}(\mathbf{x}_{in} + \text{Sublayer}(\mathbf{x}_{in})).
$$

При обратном распространении градиенты проходят через LayerNorm, которая масштабирует их. В глубоких сетях это приводит к постепенному затуханию градиентов.

В **Pre-Norm** выход слоя:

$$
\mathbf{x}_{out} = \mathbf{x}_{in} + \text{Sublayer}(\text{LayerNorm}(\mathbf{x}_{in})).
$$

Здесь градиенты от $\mathbf{x}_{out}$ к $\mathbf{x}_{in}$ распространяются по двум путям: напрямую (через остаточную связь, с коэффициентом 1) и через подслой. Прямой путь обеспечивает, что градиенты не затухают даже в очень глубоких сетях.

Это особенно важно для современных моделей, которые могут иметь до 100 и более слоёв.

---

### 6. Сравнение энкодера и декодера

| Критерий | Энкодер | Декодер |
|----------|---------|---------|
| **Количество подслоёв** | 2 | 3 |
| **Self-Attention** | Без маски (двусторонний) | С маской (односторонний, каузальный) |
| **Cross-Attention** | Отсутствует | Присутствует (Q из декодера, K,V из энкодера) |
| **Задача** | Понимание входа | Генерация выхода |
| **Параллелизация** | Полная | Ограниченная (инференс последовательный) |

---

### 7. Схема размерностей на каждом этапе

```mermaid
flowchart LR
    subgraph Encoder["Энкодер"]
        E1["X: T×d_model"] --> E2["Self-Attn: T×d_model"] --> E3["Add & Norm: T×d_model"] --> E4["FFN: T×d_model"] --> E5["Add & Norm: T×d_model"]
    end
    
    subgraph Decoder["Декодер (шаг t)"]
        D1["Y_{<t}: (t-1)×d_model"] --> D2["Masked Attn: (t-1)×d_model"] --> D3["Add & Norm: (t-1)×d_model"] --> D4["Cross-Attn: (t-1)×d_model"] --> D5["Add & Norm: (t-1)×d_model"] --> D6["FFN: (t-1)×d_model"] --> D7["Add & Norm: (t-1)×d_model"] --> D8["Проекция: vocab_size"]
    end
    
    Z["Z: T×d_model (из энкодера)"] --> D4
```

---

### 8. Заключение

Стек слоёв в энкодере и декодере Transformer представляет собой мощный механизм иерархического извлечения признаков. Энкодер последовательно уточняет представления токенов, обогащая их контекстной информацией, а декодер авторегрессивно генерирует выходную последовательность, используя как внутренние зависимости, так и информацию из энкодера. Переход от пост-нормализации к пре-нормализации стал ключевым фактором, позволившим масштабировать модели до сотен слоёв и миллиардов параметров, что лежит в основе успеха современных LLM.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Tenney, I., et al. (2019). *BERT Rediscovers the Classical NLP Pipeline*. ACL.  
   🔗 [https://arxiv.org/abs/1905.05950](https://arxiv.org/abs/1905.05950)

3. Clark, K., et al. (2019). *What Does BERT Look At? An Analysis of BERT's Attention*. BlackboxNLP Workshop.  
   🔗 [https://arxiv.org/abs/1906.04341](https://arxiv.org/abs/1906.04341)

4. Xiong, R., et al. (2020). *On Layer Normalization in the Transformer Architecture*. ICML.  
   🔗 [https://arxiv.org/abs/2002.04745](https://arxiv.org/abs/2002.04745)

## Тема 2.3. Поток данных: от входа до выхода

В предыдущих разделах мы рассмотрели отдельные компоненты архитектуры Transformer: энкодер, декодер, механизмы внимания и нормализации. Теперь настало время собрать все части в единую картину и проследить полный путь данных — от момента, когда исходный текст попадает на вход модели, до генерации выходного токена. Понимание этого потока критически важно для осознания того, как все компоненты взаимодействуют друг с другом в реальном времени.

---

### 1. Общая схема потока данных

На высоком уровне поток данных в архитектуре Encoder-Decoder можно представить следующим образом:

```mermaid
flowchart TD
    subgraph Input["ВХОД"]
        T1["Исходный текст: 'I love reading'"]
        T2["Токенизация: ['I', 'love', 'read', 'ing']"]
        T3["IDs: [42, 156, 389, 1023]"]
        T4["Эмбеддинги + PE: X ∈ ℝ^{4×512}"]
    end

    subgraph Encoder["ЭНКОДЕР (N=6 слоёв)"]
        E1["Self-Attention"]
        E2["Add & Norm"]
        E3["FFN"]
        E4["Add & Norm"]
    end

    subgraph Decoder["ДЕКОДЕР (N=6 слоёв, авторегрессивно)"]
        D1["Masked Self-Attention"]
        D2["Add & Norm"]
        D3["Cross-Attention"]
        D4["Add & Norm"]
        D5["FFN"]
        D6["Add & Norm"]
    end

    subgraph Output["ВЫХОД"]
        O1["Линейный слой: d_model → vocab_size"]
        O2["Softmax"]
        O3["Выбор токена"]
    end

    T4 --> Encoder
    Encoder -->|"Z ∈ ℝ^{4×512}"| Decoder
    Decoder --> O1 --> O2 --> O3
    O3 -->|"следующий токен"| Decoder

    style Input fill:#e3f2fd
    style Encoder fill:#f3e5f5
    style Decoder fill:#fce4ec
    style Output fill:#e0f7fa
```

На диаграмме видно ключевое свойство: **авторегрессивная петля** — выходной токен возвращается в декодер для генерации следующего.

---

### 2. Этап 1: От текста к эмбеддингам

#### 2.1. Токенизация

Первый шаг — преобразование исходного текста в последовательность токенов. Токенизация разбивает текст на минимальные смысловые единицы: слова, подслова или символы.

Пример: предложение *"I love reading"* токенизируется как `["I", "love", "read", "ing"]` (если используется BPE-токенизатор, как в оригинальной статье). Каждому токену присваивается уникальный числовой идентификатор из словаря модели. Пусть наш словарь имеет размер $V = 50000$, тогда:

$$
\text{["I", "love", "read", "ing"]} \rightarrow [42, 156, 389, 1023].
$$

#### 2.2. Эмбеддинги

Каждый идентификатор преобразуется в вектор фиксированной размерности $d_{\text{model}}$ с помощью обучаемой матрицы эмбеддингов $E \in \mathbb{R}^{V \times d_{\text{model}}}$:

$$
\mathbf{e}_i = E[\text{token\_id}_i] \in \mathbb{R}^{d_{\text{model}}}.
$$

В оригинальном Transformer $d_{\text{model}} = 512$. Таким образом, мы получаем матрицу токенных эмбеддингов:

$$
\mathbf{E}_{\text{tokens}} \in \mathbb{R}^{T \times d_{\text{model}}},
$$

где $T$ — длина последовательности (в нашем примере $T = 4$).

---

### 3. Этап 2: Позиционное кодирование

Поскольку Self-Attention не имеет встроенного понятия порядка, нам необходимо добавить информацию о позиции токена в последовательности. В оригинальной статье используется **синусоидальное позиционное кодирование**:

$$
\begin{aligned}
\mathbf{PE}_{(pos, 2i)} &= \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \\
\mathbf{PE}_{(pos, 2i+1)} &= \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right),
\end{aligned}
$$

где $pos$ — позиция токена (0, 1, 2, ...), $i$ — индекс измерения (0, 1, ..., $d_{\text{model}}/2 - 1$).

Позиционное кодирование $\mathbf{PE} \in \mathbb{R}^{T \times d_{\text{model}}}$ поэлементно суммируется с токенными эмбеддингами:

$$
\mathbf{X} = \mathbf{E}_{\text{tokens}} + \mathbf{PE} \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

Этот процесс можно проиллюстрировать следующей схемой:

```mermaid
flowchart LR
    subgraph Embedding["Формирование входа энкодера"]
        T["Токены: x₁, x₂, ..., x_T"]
        E["Матрица эмбеддингов: T×d_model"]
        PE["Позиционное кодирование: T×d_model"]
        X["Вход энкодера: X = E + PE"]
    end
    
    T --> E
    T --> PE
    E --> X
    PE --> X
    
    style T fill:#e3f2fd
    style E fill:#e8f5e9
    style PE fill:#fff3e0
    style X fill:#f3e5f5
```

---

### 4. Этап 3: Энкодер

Матрица $\mathbf{X} \in \mathbb{R}^{T \times d_{\text{model}}}$ поступает на вход стека из $N$ идентичных слоёв энкодера (в оригинале $N = 6$). Каждый слой выполняет два преобразования:

1. **Multi-Head Self-Attention** (без маски, двусторонний контекст):
   $$
   \mathbf{X}^{(1)} = \text{LayerNorm}\left(\mathbf{X} + \text{MHA}(\mathbf{X})\right).
   $$

2. **Position-wise Feed-Forward Network**:
   $$
   \mathbf{X}^{(2)} = \text{LayerNorm}\left(\mathbf{X}^{(1)} + \text{FFN}(\mathbf{X}^{(1)})\right).
   $$

После прохождения всех $N$ слоёв мы получаем **контекстуализированные представления**:

$$
\mathbf{Z} = \text{EncoderStack}(\mathbf{X}) \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

Каждый вектор $\mathbf{z}_i$ теперь содержит информацию о всей входной последовательности, обогащённую контекстом.

---

### 5. Этап 4: Декодер (авторегрессивная генерация)

Декодер генерирует выходную последовательность **токен за токеном** (авторегрессивно). На каждом шаге $t$ он получает:

- Выход энкодера $\mathbf{Z} \in \mathbb{R}^{T \times d_{\text{model}}}$;
- Уже сгенерированные токены $y_1, \dots, y_{t-1}$.

Процесс начинается со специального токена `[START]`, который подаётся на вход декодера на первом шаге.

#### 5.1. Шаг 1: Генерация первого токена

Начальная входная последовательность декодера: `[START]`.

1. Токен `[START]` преобразуется в эмбеддинг и суммируется с позиционным кодированием (позиция 0): $\mathbf{Y}_{<1} = \mathbf{e}_{\text{[START]}} + \mathbf{PE}_0$.
2. Декодер обрабатывает $\mathbf{Y}_{<1}$ через свои $N$ слоёв, используя Cross-Attention для доступа к $\mathbf{Z}$.
3. Выход декодера проходит через линейный слой и softmax, давая распределение вероятностей по словарю.
4. Выбирается токен с максимальной вероятностью — $y_1$.

#### 5.2. Шаг 2: Генерация второго токена

Теперь вход декодера: `[START, y_1]`. Оба токена преобразуются в эмбеддинги с добавлением позиционных кодирований (позиции 0 и 1), и процесс повторяется.

> **Важно:** На каждом шаге декодер **не пересчитывает энкодер**, а использует уже готовый выход $\mathbf{Z}$. Это ключевое отличие от RNN, где скрытое состояние пересчитывается с нуля для каждого токена.

#### 5.3. Авторегрессивный цикл

Этот процесс продолжается до тех пор, пока не будет сгенерирован токен `[END]` или не достигнута максимальная длина последовательности.

```mermaid
flowchart LR
    subgraph Generation["Авторегрессивная генерация"]
        S1["Шаг 1: [START] → y₁"]
        S2["Шаг 2: [START, y₁] → y₂"]
        S3["Шаг 3: [START, y₁, y₂] → y₃"]
        S4["..."]
        SE["Шаг M: ... → [END]"]
    end
    
    S1 --> S2 --> S3 --> S4 --> SE
    
    style S1 fill:#fce4ec
    style S2 fill:#fce4ec
    style S3 fill:#fce4ec
    style S4 fill:#fce4ec
    style SE fill:#e0f7fa
```

---

### 6. Этап 5: Выходной слой и выбор токена

На каждом шаге генерации декодер выдаёт скрытое состояние $\mathbf{h}_t \in \mathbb{R}^{d_{\text{model}}}$ (соответствующее последнему сгенерированному токену). Это состояние преобразуется в распределение вероятностей по словарю:

$$
\mathbf{p}_t = \text{softmax}\left(\mathbf{h}_t W_{\text{out}} + b_{\text{out}}\right),
$$

где $W_{\text{out}} \in \mathbb{R}^{d_{\text{model}} \times V}$, $V$ — размер словаря.

Выбор токена обычно осуществляется **жадно** (argmax) или с использованием **beam search** для улучшения качества.

---

### 7. Сквозной пример с размерами

Рассмотрим полный пример для конкретных значений:

- $d_{\text{model}} = 512$;
- $T = 4$ (длина входа);
- $V = 50000$ (размер словаря);
- $N = 6$ (число слоёв);

```mermaid
flowchart LR
    subgraph Dimensions["Размерности на каждом этапе"]
        A["Вход: текст 'I love reading'"]
        B["Токены: 4 токена"]
        C["IDs: [42, 156, 389, 1023]"]
        D["Эмбеддинги: 4×512"]
        E["+ PE: 4×512"]
        F["Вход энкодера X: 4×512"]
        G["Энкодер (6 слоёв): 4×512 → 4×512"]
        H["Выход энкодера Z: 4×512"]
        I["Декодер (шаг t): (t-1)×512 → 1×512"]
        J["Линейный слой: 512 → 50000"]
        K["Softmax: распределение по 50000 токенам"]
        L["Выходной токен: ID"]
    end
    
    A --> B --> C --> D --> E --> F --> G --> H --> I --> J --> K --> L
    
    style A fill:#e3f2fd
    style B fill:#e3f2fd
    style C fill:#e3f2fd
    style D fill:#e8f5e9
    style E fill:#fff3e0
    style F fill:#f3e5f5
    style G fill:#f3e5f5
    style H fill:#f3e5f5
    style I fill:#fce4ec
    style J fill:#e0f7fa
    style K fill:#e0f7fa
    style L fill:#e0f7fa
```

---

### 8. Таблица этапов с размерностями

| № | Этап | Входные данные | Размерность | Выходные данные | Размерность |
|---|------|----------------|-------------|-----------------|-------------|
| 1 | Токенизация | Исходный текст | — | Токены | $T$ |
| 2 | ID токенов | Токены | — | Числовые ID | $T$ |
| 3 | Эмбеддинги | ID токенов | $T \times 1$ | Токенные эмбеддинги | $T \times d_{\text{model}}$ |
| 4 | Позиционное кодирование | Позиции $0,\dots,T-1$ | $T \times 1$ | PE | $T \times d_{\text{model}}$ |
| 5 | Сложение | Токенные эмбеддинги + PE | $T \times d_{\text{model}}$ | Вход энкодера $\mathbf{X}$ | $T \times d_{\text{model}}$ |
| 6 | Энкодер (N слоёв) | $\mathbf{X}$ | $T \times d_{\text{model}}$ | Выход энкодера $\mathbf{Z}$ | $T \times d_{\text{model}}$ |
| 7 | Декодер (шаг t) | $y_{<t}$, $\mathbf{Z}$ | $(t-1) \times d_{\text{model}}$, $T \times d_{\text{model}}$ | Скрытое состояние $\mathbf{h}_t$ | $1 \times d_{\text{model}}$ |
| 8 | Линейный слой | $\mathbf{h}_t$ | $1 \times d_{\text{model}}$ | Логиты | $1 \times V$ |
| 9 | Softmax | Логиты | $1 \times V$ | Распределение вероятностей | $1 \times V$ |
| 10 | Выбор токена | Распределение | $1 \times V$ | ID токена $y_t$ | $1$ |

---

### 9. Заключение

Поток данных в Transformer представляет собой элегантную цепочку преобразований, где дискретные токены постепенно превращаются в непрерывные контекстуализированные представления, затем генеративно разворачиваются обратно в дискретный текст. Ключевые особенности этого потока:

1. **Параллелизм в энкодере** — все позиции обрабатываются одновременно.
2. **Авторегрессия в декодере** — генерация идёт последовательно, но с доступом ко всему входу через Cross-Attention.
3. **Неизменность размерности** — $d_{\text{model}}$ сохраняется на протяжении всей сети до выходного слоя.
4. **Повторное использование $\mathbf{Z}$** — энкодер вычисляется один раз для всей входной последовательности, что значительно ускоряет инференс по сравнению с RNN.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Alammar, J. (2018). *The Illustrated Transformer*.  
   🔗 [https://jalammar.github.io/illustrated-transformer/](https://jalammar.github.io/illustrated-transformer/)

3. Phuong, M., & Hutter, M. (2022). *Formal Algorithms for Transformers*. arXiv:2207.09238.  
   🔗 [https://arxiv.org/abs/2207.09238](https://arxiv.org/abs/2207.09238)

## Тема 3.1. Преобразование текста в числа: токенизация

Перед тем как текст может быть обработан нейронной сетью, он должен быть преобразован в числовой формат. Этот процесс, называемый **токенизацией**, является одним из наиболее фундаментальных и одновременно критически важных этапов в обработке естественного языка. Качество токенизации напрямую влияет на способность модели понимать язык, обрабатывать редкие слова и обобщать на новые данные. В этом разделе мы рассмотрим основные подходы к токенизации, их преимущества и ограничения, а также современные алгоритмы, используемые в ведущих LLM.

---

### 1. Зачем нужна токенизация

Нейронные сети, включая архитектуру Transformer, работают исключительно с числами. Они не понимают символов, слов или предложений в том виде, в каком их воспринимает человек. Задача токенизации — найти способ представления текста в виде последовательности чисел, который был бы одновременно:

1. **Компактным** — не слишком длинным для эффективной обработки;
2. **Полным** — способным представить любой возможный текст;
3. **Семантически осмысленным** — чтобы схожие по смыслу единицы имели схожие представления.

На протяжении развития NLP было предложено три основных подхода к решению этой задачи: токенизация на уровне слов, на уровне символов и на уровне подслов (субсловная токенизация).

---

### 2. Типы токенизации

#### 2.1. Word-Based (словоуровневая) токенизация

Самый интуитивный подход — разбивать текст по пробелам и знакам препинания.

**Пример:**
```
Вход: "I love cats!"
Токены: ["I", "love", "cats", "!"]
```

**Преимущества:**
- Простота реализации
- Интуитивная интерпретируемость (каждый токен — слово или знак препинания)
- Короткие последовательности (мало токенов на предложение)

**Недостатки:**
1. **Проблема неизвестных слов (OOV — Out-of-Vocabulary):** Любое слово, отсутствующее в словаре, становится неизвестным токеном `[UNK]`, что ведёт к потере информации.
2. **Огромный размер словаря:** Английский язык содержит сотни тысяч слов, а некоторые языки (например, агглютинативные) имеют практически бесконечное число словоформ.
3. **Морфологическая сложность:** Слова "run", "runs", "running", "ran" — разные токены, хотя имеют общий корень и смысл.
4. **Языки без пробелов:** В китайском, японском и корейском языках нет явных разделителей между словами, что делает словоуровневую токенизацию неприменимой.

#### 2.2. Character-Based (символьная) токенизация

Альтернативный подход — рассматривать каждый символ как отдельный токен.

**Пример:**
```
Вход: "cat"
Токены: ["c", "a", "t"]
```

**Преимущества:**
- **Отсутствие OOV:** Любое слово может быть представлено как последовательность символов.
- **Малый словарь:** Для любого языка достаточно нескольких сотен символов (включая буквы, цифры, знаки препинания).
- **Работа с любым языком:** Не требует знаний о структуре слов.

**Недостатки:**
- **Длинные последовательности:** Слово из 5 букв заменяется 5 токенами вместо одного, что увеличивает длину последовательности в 5-10 раз.
- **Потеря семантики:** Модель должна самостоятельно выучить, что "c", "a", "t" вместе образуют слово "cat". Это требует больше данных и вычислительных ресурсов.
- **Отсутствие информации о слове:** Модель не может использовать знания о том, что "cats" и "cat" — это одно и то же слово в разных формах.

#### 2.3. Subword-Based (субсловная) токенизация

Золотая середина — разбивать слова на часто встречающиеся **подслова** (субслова). Частые целые слова становятся отдельными токенами, редкие слова разбиваются на осмысленные части.

**Пример:**
```
Вход: "lower", "low", "lowest", "lowlands"
Токены: ["lo", "wer"], ["lo", "w"], ["lo", "west"], ["lo", "w", "lands"]
```

**Преимущества:**
- **Компактность:** Короткая длина последовательности (близка к словоуровневой)
- **Отсутствие OOV:** Редкие слова разбиваются на известные подслова
- **Морфологическая осмысленность:** Корни, приставки и суффиксы становятся отдельными токенами
- **Мультиязычность:** Хорошо работает для языков с богатой морфологией

Именно субсловная токенизация используется во всех современных LLM, включая GPT, BERT, LLaMA, Qwen и Mistral. Далее мы рассмотрим три основных алгоритма субсловной токенизации.

---

### 3. Алгоритм BPE (Byte-Pair Encoding)

**BPE** был предложен в 1994 году как алгоритм сжатия данных, а в 2015 году адаптирован для NLP в работе Sennrich et al. [1]. Сегодня BPE используется в GPT, LLaMA, Qwen и многих других моделях.

#### 3.1. Принцип работы

Алгоритм BPE начинает с базового словаря, содержащего все уникальные символы (или байты) в обучающем корпусе. Затем он итеративно выполняет следующие шаги:

1. Подсчитывает частоту всех соседних пар символов/подслов в корпусе.
2. Находит самую частотную пару.
3. Объединяет эту пару в новый токен.
4. Добавляет новый токен в словарь.
5. Повторяет до достижения целевого размера словаря.

#### 3.2. Пошаговый пример обучения BPE

Рассмотрим корпус из четырёх слов с частотами:

| Слово | Частота |
|-------|---------|
| "low" | 5 |
| "lower" | 2 |
| "lowest" | 2 |
| "lowlands" | 1 |

**Начальный словарь:** символы `["l", "o", "w", "e", "r", "s", "t", "a", "n", "d"]`

**Итерация 1:** Считаем частоты всех пар:
- `"lo"`: встречается в "low"(5), "lower"(2), "lowest"(2), "lowlands"(1) = **10 раз** (самая частотная!)
- `"ow"`: встречается 5+2+2+1 = 10 раз
- `"we"`: в "lower" 2 раза
- и т.д.

Выбираем `"lo"` (или `"ow"` — при равной частоте алгоритм может выбрать любой). Объединяем: `"lo"` становится новым токеном. Слова теперь выглядят так: `["lo", "w"]`, `["lo", "wer"]`, `["lo", "west"]`, `["lo", "wlands"]`.

**Итерация 2:** Снова считаем частоты пар:
- `"lo" + "w"`: встречается 5+1 = 6 раз (в "low" и "lowlands")
- `"we"`: в "lower" 2 раза
- `"es"`: в "lowest" 2 раза
- `"we"` и `"es"` имеют частоту 2
- и т.д.

После 10 итераций словарь может выглядеть так:
`["l", "o", "w", "e", "r", "s", "t", "a", "n", "d", "lo", "ow", "we", "es", "st", "and"]`

#### 3.3. Кодирование нового текста

Для кодирования нового текста алгоритм применяет правила слияния в том же порядке, в котором они были выучены:

```
"lowlands" → ["lo", "w", "lands"]
"lower" → ["lo", "wer"]
"lowest" → ["lo", "west"]
```

#### 3.4. Byte-Level BPE (GPT-2, GPT-4)

Классический BPE оперирует символами Unicode. Проблема: если в тексте встречается символ, отсутствовавший в обучающем корпусе, он становится неизвестным. В **Byte-Level BPE** (предложен в GPT-2) базовый словарь состоит из 256 байтов, а не символов. Это гарантирует, что **любой текст может быть закодирован**, включая эмодзи, редкие символы и тексты на любых языках.

---

### 4. WordPiece (BERT)

**WordPiece** был разработан в Google и используется в BERT, DistilBERT и других моделях. Он очень похож на BPE, но использует другую функцию для выбора пары для объединения.

#### 4.1. Отличие от BPE

Вместо частоты пар, WordPiece использует **вероятностную оценку**:

$$
\text{Score}(a, b) = \frac{\text{freq}(a, b)}{\text{freq}(a) \cdot \text{freq}(b)}
$$

Это отношение показывает, насколько появление $b$ после $a$ вероятнее, чем можно было бы ожидать из независимых частот. Чем выше это отношение, тем сильнее связаны два токена.

#### 4.2. Пример

Для слова "playing":
- WordPiece: `["play", "##ing"]`
- Символ `##` указывает, что токен является продолжением предыдущего (не началом нового слова)

#### 4.3. Кодирование

WordPiece использует **жадный алгоритм**: начиная с начала слова, он находит самый длинный токен из словаря, затем переходит к остатку слова.

```
"tokenization" → "token" (в словаре есть) → "##ization" (в словаре есть)
```

**Критическое ограничение:** Если какая-то часть слова не может быть закодирована токенами из словаря, весь токен становится `[UNK]`. Это делает WordPiece менее устойчивым к OOV, чем BPE.

---

### 5. SentencePiece (LLaMA, Qwen, Mistral)

**SentencePiece** — это не алгоритм, а фреймворк, реализующий алгоритм **Unigram** (а также BPE) [2]. Он используется в LLaMA, Qwen, Mistral и других современных моделях.

#### 5.1. Особенности SentencePiece

1. **Не требует пробелов:** Работает с сырым текстом, кодируя пробелы специальным символом `▁` (U+2581).
2. **Unigram алгоритм:** Начинает с большого словаря (все возможные подслова) и итеративно удаляет наименее важные токены.
3. **Мультиязычность:** Оптимизирован для работы с разными языками, включая японский и китайский.

#### 5.2. Unigram Language Model

Unigram использует вероятностную модель: каждый токен имеет вероятность $p(t)$. Вероятность сегментации слова — произведение вероятностей токенов. Алгоритм обучения:

1. Начинаем с очень большого словаря (все подслова, встречающиеся в корпусе).
2. Итеративно удаляем токены, удаление которых меньше всего ухудшает правдоподобие корпуса.
3. Повторяем до достижения целевого размера словаря.

```mermaid
flowchart TD
    A["Начальный словарь: все подслова"] --> B["Оценка важности каждого токена"]
    B --> C["Удаление наименее важных токенов"]
    C --> D["Достигнут целевой размер?"]
    D -->|Нет| B
    D -->|Да| E["Финальный словарь"]
    
    style A fill:#e3f2fd
    style B fill:#fff3e0
    style C fill:#ffcdd2
    style D fill:#f3e5f5
    style E fill:#e0f7fa
```

#### 5.3. Параметр character_coverage

SentencePiece поддерживает параметр `character_coverage`, который определяет, какая доля символов должна быть покрыта словарём. Для мультиязычных моделей рекомендуется значение `0.9995`, чтобы покрыть 99.95% всех символов во всех языках.

---

### 6. Сравнительный анализ методов

| Метод | Словарь | Плюсы | Минусы | Модели |
|-------|---------|-------|--------|--------|
| Word-Based | Сотни тысяч | Простота, короткие последовательности | OOV, огромный словарь | Устаревшие системы |
| Character-Based | Сотни | Нет OOV, малый словарь | Длинные последовательности, потеря семантики | Быстрое прототипирование |
| BPE | 30-100K | Нет OOV, морфологическая осмысленность | Требует предварительной токенизации по пробелам | GPT, LLaMA, Qwen, Mistral |
| WordPiece | 30K | Хорош для английского | OOV при неизвестных подсловах, `[UNK]` | BERT, DistilBERT |
| SentencePiece | 30-100K | Мультиязычный, нет пробелов | Сложнее в настройке | LLaMA, Qwen, T5, mT5 |

---

### 7. Практический пример с Hugging Face

Ниже приведён код, демонстрирующий работу токенизатора BERT (WordPiece) на Python:

```python
from transformers import AutoTokenizer

# Загрузка токенизатора BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Исходное предложение
text = "I love reading books!"

# Токенизация
tokens = tokenizer.tokenize(text)
print("Токены:", tokens)

# Преобразование в IDs
ids = tokenizer.convert_tokens_to_ids(tokens)
print("IDs:", ids)

# Восстановление текста из IDs
decoded = tokenizer.decode(ids)
print("Декодированный текст:", decoded)
```

**Результат выполнения:**
```
Токены: ['i', 'love', 'read', '##ing', 'books', '!']
IDs: [1045, 2293, 2294, 2106, 2612, 999]
Декодированный текст: "i love reading books!"
```

Обратите внимание:
- Слово "reading" разбито на `["read", "##ing"]` — это классический пример WordPiece.
- Текст приведён к нижнему регистру (это особенность `bert-base-uncased`).

#### Визуализация процесса токенизации:

```mermaid
flowchart LR
    A["'I love reading books!'"] --> B["Токенизатор BERT"]
    B --> C["['i', 'love', 'read', '##ing', 'books', '!']"]
    C --> D["[1045, 2293, 2294, 2106, 2612, 999]"]
    D --> E["Нейронная сеть"]
    
    style A fill:#e3f2fd
    style B fill:#fff3e0
    style C fill:#e8f5e9
    style D fill:#f3e5f5
    style E fill:#e0f7fa
```

---

### 8. Сравнение размеров словарей популярных моделей

| Модель | Алгоритм | Размер словаря | Особенности |
|--------|----------|----------------|-------------|
| BERT-base | WordPiece | 30,522 | Специальные токены: `[CLS]`, `[SEP]`, `[MASK]` |
| GPT-2 | Byte-level BPE | 50,257 | Базовый словарь — 256 байтов |
| GPT-3/GPT-4 | Byte-level BPE | 50,257 | Масштабируемый, без OOV |
| LLaMA (Meta) | SentencePiece (BPE) | 32,000 | Мультиязычный |
| Qwen 2.5 | SentencePiece (BPE) | 151,936 | Большой словарь для мультиязычности |
| Mistral | SentencePiece (BPE) | 32,000 | Оптимизирован для эффективности |
| T5 | SentencePiece (Unigram) | 32,000 | Текст-в-текст для всех задач |

---

### 9. Заключение

Токенизация — это первый и критически важный этап обработки текста в LLM. Выбор подхода определяет:

- **Способность модели работать с редкими словами** (BPE и SentencePiece решают эту проблему)
- **Эффективность использования памяти и вычислительных ресурсов** (длина последовательности)
- **Мультиязычные возможности** (SentencePiece не требует пробелов)

Современные модели практически повсеместно используют субсловную токенизацию с байтовым уровнем (Byte-Level BPE или SentencePiece), что обеспечивает баланс между компактностью и полнотой покрытия. Понимание этих алгоритмов необходимо не только для правильной загрузки предобученных моделей, но и для обучения собственных моделей на новых языках или специфических доменах.

---

### Литература

1. Sennrich, R., Haddow, B., & Birch, A. (2015). *Neural Machine Translation of Rare Words with Subword Units*. ACL.  
   🔗 [https://arxiv.org/abs/1508.07909](https://arxiv.org/abs/1508.07909)

2. Kudo, T., & Richardson, J. (2018). *SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing*. EMNLP.  
   🔗 [https://arxiv.org/abs/1808.06226](https://arxiv.org/abs/1808.06226)

3. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

4. Radford, A., et al. (2019). *Language Models are Unsupervised Multitask Learners*. OpenAI.  
   🔗 [https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

## Тема 3.2. Эмбеддинги — векторное представление токенов

После того как текст был преобразован в последовательность числовых идентификаторов токенов, встаёт следующая задача: как представить эти дискретные символы в форме, пригодной для обработки нейронной сетью? Решением являются **эмбеддинги** (embeddings) — плотные векторные представления, которые отображают каждый токен в непрерывное многомерное пространство. В этом разделе мы рассмотрим математические основы эмбеддингов, их свойства, способы обучения и практическое применение в архитектуре Transformer.

---

### 1. Определение и математическая формализация

**Эмбеддинг** (или векторное представление) — это отображение дискретного объекта (токена) в непрерывное векторное пространство фиксированной размерности. В контексте Transformer эмбеддинги представляют собой обучаемую матрицу:

$$
\mathbf{E} \in \mathbb{R}^{V \times d_{\text{model}}},
$$

где $V$ — размер словаря (количество уникальных токенов), а $d_{\text{model}}$ — размерность эмбеддингов. Каждая строка этой матрицы соответствует одному токену и содержит его векторное представление.

Преобразование токена в эмбеддинг представляет собой операцию индексирования:

$$
\mathbf{e}_i = \mathbf{E}[\text{token\_id}_i] \in \mathbb{R}^{d_{\text{model}}},
$$

где $\text{token\_id}_i$ — числовой идентификатор $i$-го токена в последовательности.

```mermaid
flowchart LR
    subgraph EmbeddingMatrix["Матрица эмбеддингов E ∈ ℝ^{V×d_model}"]
        direction TB
        E1["Токен 0: [0.12, 0.45, -0.33, ...]"]
        E2["Токен 1: [0.78, -0.21, 0.56, ...]"]
        E3["Токен 2: [-0.44, 0.91, 0.12, ...]"]
        EDot["..."]
        EV["Токен V-1: [...]"]
    end
    
    ID["token_id = 42"] --> E1
    
    style ID fill:#e3f2fd
    style EmbeddingMatrix fill:#f3e5f5
```

#### 1.1. Размерность эмбеддингов

Выбор размерности $d_{\text{model}}$ — важный гиперпараметр, определяющий выразительную способность модели и её вычислительную сложность. В оригинальном Transformer $d_{\text{model}} = 512$, но в современных моделях используются значительно большие размерности:

| Модель | $d_{\text{model}}$ | $V$ (размер словаря) | Год |
|--------|-------------------|---------------------|-----|
| Transformer (base) | 512 | 37,000 | 2017 |
| BERT-base | 768 | 30,522 | 2018 |
| GPT-2 | 768 | 50,257 | 2019 |
| BERT-large | 1024 | 30,522 | 2019 |
| GPT-3 (175B) | 12288 | 50,257 | 2020 |
| LLaMA 3 (8B) | 4096 | 128,256 | 2024 |
| LLaMA 3 (70B) | 8192 | 128,256 | 2024 |
| Qwen 2.5 (72B) | 8192 | 151,936 | 2024 |

---

### 2. Математические свойства эмбеддингов

#### 2.1. Семантическая близость

Фундаментальное свойство эмбеддингов заключается в том, что **семантически близкие токены имеют близкие векторы**. Мера близости обычно вычисляется как **косинусное расстояние**:

$$
\text{cos\_sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|} = \frac{\sum_{i=1}^{d} a_i b_i}{\sqrt{\sum_{i=1}^{d} a_i^2} \cdot \sqrt{\sum_{i=1}^{d} b_i^2}}.
$$

Косинусное расстояние принимает значения от $-1$ до $1$, где $1$ означает идентичные векторы (максимальное сходство), $0$ — ортогональные (независимые), а $-1$ — противоположные.

**Числовой пример:**

Пусть у нас есть три вектора (в упрощённой размерности $d=3$):

$$
\begin{aligned}
\mathbf{v}_{\text{cat}} &= [0.8, 0.1, 0.3], \\
\mathbf{v}_{\text{cat}} &= [0.7, 0.2, 0.4] \quad (\text{другой кот}), \\
\mathbf{v}_{\text{dog}} &= [0.2, 0.9, 0.1], \\
\mathbf{v}_{\text{table}} &= [0.1, 0.1, 0.9].
\end{aligned}
$$

Вычислим косинусное расстояние:

$$
\text{cos\_sim}(\text{cat}_1, \text{cat}_2) \approx 0.95 \quad (\text{высокое сходство}),
$$

$$
\text{cos\_sim}(\text{cat}, \text{dog}) \approx 0.35 \quad (\text{среднее сходство}),
$$

$$
\text{cos\_sim}(\text{cat}, \text{table}) \approx 0.12 \quad (\text{низкое сходство}).
$$

#### 2.2. Аналоговые отношения

Одно из наиболее удивительных свойств эмбеддингов — способность кодировать **аналоговые отношения** через векторную арифметику. Если векторы слов обучены качественно, то:

$$
\mathbf{w}_{\text{king}} - \mathbf{w}_{\text{man}} + \mathbf{w}_{\text{woman}} \approx \mathbf{w}_{\text{queen}}.
$$

Это означает, что вектор «королевской власти» ($\mathbf{w}_{\text{king}} - \mathbf{w}_{\text{man}}$) добавляется к вектору «женщины» и даёт вектор «королевы».

**Другие примеры аналогий:**

| Отношение | Аналогия | Результат |
|-----------|----------|-----------|
| Страна-столица | Paris - France + Italy | ≈ Rome |
| Единственное-множественное | cat - cats + dogs | ≈ dog |
| Настоящее-прошедшее | run - ran + swam | ≈ swim |
| Род занятий | doctor - hospital + school | ≈ teacher |

```mermaid
flowchart LR
    subgraph Analogies["Векторные аналогии"]
        direction LR
        K["w_king"] --> M["- w_man"]
        M --> W["+ w_woman"]
        W --> Q["≈ w_queen"]
    end
    
    style K fill:#e3f2fd
    style M fill:#fff3e0
    style W fill:#e8f5e9
    style Q fill:#f3e5f5
```

#### 2.3. Линейные свойства

Эмбеддинги обладают свойством **линейности**: семантические отношения часто могут быть выражены как линейные комбинации векторов. Это свойство лежит в основе методов аналогий и делает эмбеддинги интерпретируемыми.

---

### 3. Обучение эмбеддингов

#### 3.1. Статичные эмбеддинги

Первые методы обучения эмбеддингов создавали **статичные** представления — один фиксированный вектор для каждого слова, независимо от контекста.

**Word2Vec (Mikolov et al., 2013) [1]:**
- Два подхода: **CBOW** (предсказание слова по контексту) и **Skip-gram** (предсказание контекста по слову).
- Размерность: обычно 100-300.
- Обучается на больших корпусах (миллиарды слов).

**GloVe (Pennington et al., 2014) [2]:**
- Использует глобальную статистику совместной встречаемости слов.
- Комбинирует преимущества матричных факторизаций и локальных контекстных методов.

**FastText (Bojanowski et al., 2016) [3]:**
- Учитывает морфологию: каждый токен представляется как сумма эмбеддингов его n-грамм символов.
- Лучше работает с редкими словами.

#### 3.2. Контекстуальные эмбеддинги

Главное ограничение статичных эмбеддингов — омонимия: слово «банк» имеет один вектор независимо от того, идёт ли речь о финансовом учреждении или о береге реки.

**Контекстуальные эмбеддинги** решают эту проблему, генерируя вектор для токена **на основе его окружения**.

**ELMo (Peters et al., 2018) [4]:**
- Использует двунаправленный LSTM.
- Генерирует разные векторы для одного слова в разных контекстах.

**BERT (Devlin et al., 2018) [5] и GPT (Radford et al., 2018) [6]:**
- Используют архитектуру Transformer.
- Обучаются на масштабных корпусах с миллиардами токенов.
- Вектор токена зависит от всего предложения (BERT) или от предыдущих токенов (GPT).

#### 3.3. Обучение эмбеддингов в Transformer

В архитектуре Transformer эмбеддинги обучаются **совместно с остальной моделью** в процессе предобучения. Это означает, что матрица эмбеддингов $\mathbf{E}$ обновляется на каждом шаге градиентного спуска вместе с весами внимания и FFN.

Математически, если $\mathcal{L}$ — функция потерь, то:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{E}} = \sum_{i=1}^{T} \frac{\partial \mathcal{L}}{\partial \mathbf{e}_i},
$$

где $\mathbf{e}_i$ — эмбеддинг $i$-го токена. Это позволяет эмбеддингам адаптироваться под конкретные задачи.

В некоторых моделях (например, GPT) матрица эмбеддингов и матрица выходного линейного слоя **разделяют веса** (weight tying), что уменьшает число параметров и улучшает обучение.

---

### 4. Визуализация эмбеддингов

Поскольку эмбеддинги имеют высокую размерность (сотни или тысячи измерений), их невозможно визуализировать напрямую. Для этого используются методы понижения размерности.

#### 4.1. t-SNE (t-Distributed Stochastic Neighbor Embedding)

t-SNE проецирует высокоразмерные данные в 2D или 3D пространство, сохраняя локальные структуры. Результат обычно показывает чёткие кластеры слов по смыслу.

```mermaid
flowchart TD
    subgraph TSV["Визуализация эмбеддингов (t-SNE)"]
        direction TB
        C1["● Кластер: Животные<br/>(cat, dog, horse)"]
        C2["● Кластер: Глаголы движения<br/>(run, walk, swim)"]
        C3["● Кластер: Еда<br/>(apple, bread, meat)"]
        C4["● Кластер: Транспорт<br/>(car, train, plane)"]
    end
    
    style C1 fill:#e3f2fd
    style C2 fill:#f3e5f5
    style C3 fill:#e8f5e9
    style C4 fill:#fff3e0
```

#### 4.2. PCA (Principal Component Analysis)

PCA находит направления максимальной дисперсии в данных и проецирует на них. Результат менее точен для кластеризации, но лучше сохраняет глобальную структуру.

---

### 5. Практический пример на Python

Ниже приведён код, демонстрирующий работу с эмбеддингами:

```python
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import KeyedVectors
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Загрузка предобученных эмбеддингов (например, FastText или GloVe)
# Для примера будем использовать синтетические векторы

# Синтетические эмбеддинги (в реальности загружаются из файла)
embeddings = {
    "king": np.array([0.9, 0.1, 0.8]),
    "man": np.array([0.8, 0.9, 0.2]),
    "woman": np.array([0.2, 0.8, 0.9]),
    "queen": np.array([0.3, 0.1, 0.9]),
    "cat": np.array([0.8, 0.2, 0.1]),
    "dog": np.array([0.7, 0.3, 0.2]),
}

# 1. Вычисление косинусного расстояния
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Косинусное расстояние (queen, woman):",
      cosine_sim(embeddings["queen"], embeddings["woman"]))
print("Косинусное расстояние (queen, cat):",
      cosine_sim(embeddings["queen"], embeddings["cat"]))

# 2. Аналоговые отношения
# w_king - w_man + w_woman ≈ w_queen
analogy = embeddings["king"] - embeddings["man"] + embeddings["woman"]
print("Вектор аналогии:", analogy)

# Проверка: какой вектор ближе к результату?
best_word = None
best_score = -1
for word, vec in embeddings.items():
    if word not in ["king", "man", "woman"]:
        score = cosine_sim(analogy, vec)
        if score > best_score:
            best_score = score
            best_word = word

print(f"Ближайший вектор к 'king - man + woman': {best_word} (сходство: {best_score:.3f})")

# 3. Визуализация t-SNE
words = list(embeddings.keys())
vectors = np.array([embeddings[w] for w in words])

tsne = TSNE(n_components=2, random_state=42)
vectors_2d = tsne.fit_transform(vectors)

plt.figure(figsize=(10, 8))
for i, word in enumerate(words):
    x, y = vectors_2d[i, 0], vectors_2d[i, 1]
    plt.scatter(x, y, s=100)
    plt.annotate(word, (x, y), fontsize=12)

plt.title("Визуализация эмбеддингов (t-SNE)")
plt.xlabel("Измерение 1")
plt.ylabel("Измерение 2")
plt.grid(True)
plt.show()
```

**Пример вывода:**
```
Косинусное расстояние (queen, woman): 0.912
Косинусное расстояние (queen, cat): 0.234
Ближайший вектор к 'king - man + woman': queen (сходство: 0.987)
```

---

### 6. Примеры семантических аналогий

| Отношение | Аналогия | Ожидаемый результат | Результат в GloVe |
|-----------|----------|-------------------|-------------------|
| Страна-столица | Berlin - Germany + France | Paris | Paris |
| Род-вид | cat - animal + dog | canine | dog |
| Единственное-множественное | car - cars + dogs | dog | dog |
| Настоящее-прошедшее | run - ran + swam | swim | swim |
| Род занятий | doctor - hospital + school | teacher | teacher |

---

### 7. Заключение

Эмбеддинги являются мостом между дискретными токенами и непрерывным пространством, в котором работает нейронная сеть. Их ключевые свойства — семантическая близость, линейные отношения и контекстуальная зависимость — делают их мощным инструментом для представления языка. В архитектуре Transformer эмбеддинги обучаются совместно с остальной моделью, что позволяет им адаптироваться под конкретные задачи и языковые особенности. Понимание эмбеддингов необходимо для эффективной работы с любыми LLM, от тонкой настройки до интерпретации результатов.

---

### Литература

1. Mikolov, T., et al. (2013). *Efficient Estimation of Word Representations in Vector Space*. ICLR.  
   🔗 [https://arxiv.org/abs/1301.3781](https://arxiv.org/abs/1301.3781)

2. Pennington, J., Socher, R., & Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation*. EMNLP.  
   🔗 [https://nlp.stanford.edu/pubs/glove.pdf](https://nlp.stanford.edu/pubs/glove.pdf)

3. Bojanowski, P., et al. (2016). *Enriching Word Vectors with Subword Information*. arXiv:1607.04606.  
   🔗 [https://arxiv.org/abs/1607.04606](https://arxiv.org/abs/1607.04606)

4. Peters, M., et al. (2018). *Deep contextualized word representations*. NAACL.  
   🔗 [https://arxiv.org/abs/1802.05365](https://arxiv.org/abs/1802.05365)

5. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

6. Radford, A., et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.  
   🔗 [https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

## Тема 3.3. Почему эмбеддинги работают: семантические свойства векторных представлений

Эмбеддинги — это не просто произвольное отображение токенов в числовые векторы. Их эффективность основана на фундаментальных свойствах языка и математических закономерностях, которые делают векторное пространство семантически осмысленным. В этом разделе мы рассмотрим теоретические основы, объясняющие, почему эмбеддинги столь эффективны, и как эти свойства проявляются на практике.

---

### 1. Распределённая семантика: гипотеза, лежащая в основе

#### 1.1. Distributional Hypothesis

Фундаментальная идея, на которой основаны все методы обучения эмбеддингов, формулируется как **гипотеза распределённой семантики** (Distributional Hypothesis), предложенная Harris (1954) и развитая Firth (1957):

> *"Words that occur in similar contexts tend to have similar meanings."*

Или, в более формальной формулировке:

> *"You shall know a word by the company it keeps."* (Firth, 1957)

Эта гипотеза утверждает, что семантика слова может быть восстановлена из статистики его употребления: слова, которые встречаются в окружении похожих слов, имеют схожие значения.

**Математическая формулировка:**

Пусть $w$ — слово, а $c$ — контекст (окружающие слова). Тогда семантика слова определяется распределением вероятностей:

$$
p(w \mid \text{context})
$$

Два слова $w_1$ и $w_2$ семантически близки, если:

$$
p(w_1 \mid \text{context}) \approx p(w_2 \mid \text{context})
$$

для всех возможных контекстов.

#### 1.2. От статистики к эмбеддингам

Эмбеддинги реализуют эту гипотезу, отображая слова в векторное пространство таким образом, что:

$$
\cos(\mathbf{v}_{w_1}, \mathbf{v}_{w_2}) \propto \text{сходство контекстов}
$$

Другими словами, чем чаще слова встречаются в похожих контекстах, тем ближе их векторы в пространстве эмбеддингов.

```mermaid
flowchart LR
    subgraph Context["Контекстное распределение"]
        W1["word₁: контекст A,B,C"]
        W2["word₂: контекст A,B,C"]
        W3["word₃: контекст D,E,F"]
    end
    
    subgraph EmbeddingSpace["Пространство эмбеддингов"]
        V1["● word₁"]
        V2["● word₂"]
        V3["● word₃"]
    end
    
    W1 --> V1
    W2 --> V2
    W3 --> V3
    
    V1 -.->|близко| V2
    V1 -.->|далеко| V3
    
    style Context fill:#e3f2fd
    style EmbeddingSpace fill:#f3e5f5
```

**Пример:** Слова «кот» и «кошка» встречаются в похожих контекстах («пушистый __», «__ мяукает»), поэтому их векторы близки. Слово «компьютер» встречается в других контекстах, поэтому его вектор далёк.

---

### 2. Косинусное расстояние: мера семантической близости

Для измерения близости векторов в пространстве эмбеддингов используется **косинусное расстояние** (cosine similarity). Это предпочтительная метрика, поскольку она:

1. Инвариантна к масштабу вектора (зависит только от направления);
2. Нормирована на интервал $[-1, 1]$;
3. Имеет чёткую семантическую интерпретацию.

#### 2.1. Формула и интерпретация

Для двух векторов $\mathbf{a}, \mathbf{b} \in \mathbb{R}^d$ косинусное расстояние определяется как:

$$
\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|} = \frac{\sum_{i=1}^{d} a_i b_i}{\sqrt{\sum_{i=1}^{d} a_i^2} \cdot \sqrt{\sum_{i=1}^{d} b_i^2}}.
$$

**Интерпретация значений:**

| Значение | Интерпретация | Семантический смысл |
|----------|---------------|---------------------|
| $\cos(\theta) = 1$ | Векторы сонаправлены | Идентичные или почти идентичные слова |
| $\cos(\theta) > 0.7$ | Высокое сходство | Синонимы или тематически близкие слова |
| $\cos(\theta) \approx 0$ | Векторы ортогональны | Семантически независимые слова |
| $\cos(\theta) < 0$ | Векторы противоположны | Антонимы или противопоставленные понятия |
| $\cos(\theta) = -1$ | Векторы противоположно направлены | Противоположные по смыслу слова |

#### 2.2. Практический пример

Рассмотрим несколько векторов в упрощённом 2D-пространстве (в реальности размерность сотни или тысячи):

| Слово | Вектор |
|-------|--------|
| cat | $[0.8, 0.6]$ |
| dog | $[0.7, 0.5]$ |
| car | $[0.1, 0.9]$ |
| house | $[0.2, 0.1]$ |

**Вычисления:**

- $\cos(\text{cat}, \text{dog}) \approx 0.99$ — очень близки (оба домашние животные)
- $\cos(\text{cat}, \text{car}) \approx 0.76$ — среднее сходство (оба могут быть объектами, но разные категории)
- $\cos(\text{cat}, \text{house}) \approx 0.70$ — среднее сходство

---

### 3. Векторные аналогии: линейные отношения в пространстве

Одно из наиболее впечатляющих свойств эмбеддингов — способность кодировать **линейные отношения** между словами. Это означает, что семантические отношения могут быть выражены как векторные операции.

#### 3.1. Формула аналогий

Если слова $w_a$, $w_b$, $w_c$, $w_d$ связаны отношением:

$$
w_a : w_b \quad \text{как} \quad w_c : w_d,
$$

то в пространстве эмбеддингов выполняется приближённое равенство:

$$
\mathbf{v}_{w_b} - \mathbf{v}_{w_a} \approx \mathbf{v}_{w_d} - \mathbf{v}_{w_c},
$$

или, что эквивалентно:

$$
\mathbf{v}_{w_d} \approx \mathbf{v}_{w_b} - \mathbf{v}_{w_a} + \mathbf{v}_{w_c}.
$$

#### 3.2. Классические примеры аналогий

| Отношение | Аналогия | Векторная операция | Результат |
|-----------|----------|-------------------|-----------|
| Страна-столица | France → Paris | $\mathbf{v}_{Paris} - \mathbf{v}_{France}$ | Столица |
| Страна-столица | Italy → Rome | $\mathbf{v}_{Rome} \approx \mathbf{v}_{Paris} - \mathbf{v}_{France} + \mathbf{v}_{Italy}$ | Rome |
| Гендер | man → king | $\mathbf{v}_{king} - \mathbf{v}_{man}$ | Мужской монарх |
| Гендер | woman → queen | $\mathbf{v}_{queen} \approx \mathbf{v}_{king} - \mathbf{v}_{man} + \mathbf{v}_{woman}$ | queen |
| Единственное-множественное | cat → cats | $\mathbf{v}_{cats} - \mathbf{v}_{cat}$ | Множественное число |
| Единственное-множественное | dog → dogs | $\mathbf{v}_{dogs} \approx \mathbf{v}_{cats} - \mathbf{v}_{cat} + \mathbf{v}_{dog}$ | dogs |

#### 3.3. Математическое обоснование

Почему векторные аналогии работают? Причина в том, что эмбеддинги обучаются предсказывать контекст. Если слова $w_a$ и $w_b$ различаются по одному семантическому признаку (например, по гендеру), то разность их векторов кодирует этот признак. Применение этой разности к другому слову позволяет «перенести» признак.

```mermaid
flowchart LR
    subgraph Gender["Гендерное отношение"]
        M["man"] --> K["king"]
        W["woman"] --> Q["queen"]
    end
    
    subgraph Vector["Векторная операция"]
        V1["v_king - v_man"] --> V2["≈ v_queen - v_woman"]
    end
    
    V2 --> Result["v_queen ≈ v_king - v_man + v_woman"]
    
    style Gender fill:#e3f2fd
    style Vector fill:#f3e5f5
    style Result fill:#e0f7fa
```

#### 3.4. Код на Python для аналогий

```python
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_analogy(word_a, word_b, word_c, embeddings, top_n=1):
    """
    Находит слово d, такое что a:b как c:d.
    d = b - a + c
    """
    # Получаем векторы
    v_a = embeddings[word_a]
    v_b = embeddings[word_b]
    v_c = embeddings[word_c]
    
    # Вычисляем целевой вектор
    v_target = v_b - v_a + v_c
    
    # Нормализуем
    v_target = v_target / np.linalg.norm(v_target)
    
    # Ищем ближайший вектор
    results = []
    for word, vec in embeddings.items():
        if word in [word_a, word_b, word_c]:
            continue
        vec_norm = vec / np.linalg.norm(vec)
        sim = np.dot(v_target, vec_norm)
        results.append((word, sim))
    
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

# Пример использования
embeddings = {
    "king": np.array([0.9, 0.1, 0.8]),
    "man": np.array([0.8, 0.9, 0.2]),
    "woman": np.array([0.2, 0.8, 0.9]),
    "queen": np.array([0.3, 0.1, 0.9]),
    "prince": np.array([0.7, 0.3, 0.7]),
    "princess": np.array([0.4, 0.2, 0.8]),
}

analogy = find_analogy("king", "man", "woman", embeddings)
print(f"king - man + woman ≈ {analogy[0][0]} (score: {analogy[0][1]:.3f})")
```

**Результат:**
```
king - man + woman ≈ queen (score: 0.987)
```

---

### 4. Кластеризация: группы слов по смыслу

Эмбеддинги естественным образом группируются в кластеры, соответствующие семантическим категориям. Это свойство широко используется для:

1. **Визуализации** структуры языка;
2. **Классификации** текстов;
3. **Поиска** семантически близких слов;
4. **Выявления** тем в документах.

#### 4.1. Визуализация кластеров

```mermaid
flowchart TD
    subgraph Clusters["Кластеры эмбеддингов (t-SNE проекция)"]
        direction TB
        C1["● Животные<br/>cat, dog, horse, cow"]
        C2["● Глаголы движения<br/>run, walk, swim, fly"]
        C3["● Еда<br/>apple, bread, meat, rice"]
        C4["● Транспорт<br/>car, train, plane, bus"]
        C5["● Мебель<br/>table, chair, bed, sofa"]
    end
    
    style C1 fill:#e3f2fd
    style C2 fill:#f3e5f5
    style C3 fill:#e8f5e9
    style C4 fill:#fff3e0
    style C5 fill:#fce4ec
```

#### 4.2. Интерпретация кластеров

Кластеры в пространстве эмбеддингов часто соответствуют:

- **Семантическим категориям:** животные, транспорт, еда;
- **Синтаксическим категориям:** глаголы, существительные, прилагательные;
- **Стилистическим категориям:** формальная vs неформальная лексика;
- **Тематическим категориям:** медицина, юриспруденция, техника.

---

### 5. Ограничения статичных эмбеддингов

Несмотря на впечатляющие свойства, статичные эмбеддинги (Word2Vec, GloVe, FastText) имеют фундаментальное ограничение.

#### 5.1. Проблема многозначности (Polysemy)

Статичные эмбеддинги присваивают **один вектор** каждому слову, независимо от контекста. Это приводит к проблеме многозначности (polysemy): слово с несколькими значениями имеет один вектор, который является усреднением всех значений.

**Пример:** Слово «bank»:

1. Финансовое учреждение: *"I deposited money in the bank."*
2. Берег реки: *"The boat reached the bank."*

В статичных эмбеддингах оба значения «bank» будут представлены одним вектором, который не отражает ни одно из значений полностью.

```mermaid
flowchart LR
    subgraph Static["Статичные эмбеддинги"]
        S["bank: один вектор для всех значений"]
    end
    
    subgraph Contextual["Контекстуальные эмбеддинги"]
        C1["bank (финансовый): вектор₁"]
        C2["bank (берег): вектор₂"]
    end
    
    S -.->|смешанное значение| C1
    S -.->|смешанное значение| C2
    
    style Static fill:#ffcdd2
    style Contextual fill:#e8f5e9
```

#### 5.2. Как Transformer решает эту проблему

Архитектура Transformer с её контекстуальными эмбеддингами решает проблему многозначности. Вместо статичного вектора для каждого токена, Transformer генерирует вектор **на основе контекста**:

$$
\mathbf{e}_i = f(\text{token}_i, \text{context}),
$$

где $f$ — функция, реализуемая стёком слоёв энкодера (Self-Attention + FFN). Это означает, что один и тот же токен «bank» будет иметь разные векторы в разных предложениях.

**Пример в Transformer:**

| Предложение | Вектор для "bank" |
|-------------|-------------------|
| *"I deposited money in the bank."* | Вектор близкий к финансовым терминам |
| *"The boat reached the bank."* | Вектор близкий к географическим терминам |

#### 5.3. Сравнение статичных и контекстуальных эмбеддингов

| Характеристика | Статичные (Word2Vec, GloVe) | Контекстуальные (BERT, GPT) |
|----------------|----------------------------|------------------------------|
| Вектор зависит от контекста | Нет | Да |
| Решает проблему многозначности | Нет | Да |
| Вычислительная сложность | Низкая (индексирование) | Высокая (проход через модель) |
| Обучение | Отдельно от основной модели | Совместно с моделью |
| Скорость инференса | Очень высокая | Зависит от размера модели |
| Применение | Быстрые системы, прототипирование | Глубокое понимание языка |

---

### 6. Заключение

Эмбеддинги работают потому, что они улавливают фундаментальную закономерность языка: **слова, встречающиеся в похожих контекстах, имеют схожие значения**. Это наблюдение, известное как гипотеза распределённой семантики, лежит в основе всех современных методов представления текста.

Ключевые свойства эмбеддингов:
1. **Семантическая близость** — измеряется через косинусное расстояние;
2. **Векторные аналогии** — семантические отношения выражаются как линейные операции;
3. **Кластеризация** — слова из одной темы естественным образом группируются.

Переход от статичных эмбеддингов к контекстуальным (как в Transformer) стал революционным шагом, позволившим решить проблему многозначности и значительно улучшить качество всех NLP-задач. Понимание этих принципов необходимо для эффективного использования и дообучения современных LLM.

---

### Литература

1. Harris, Z. (1954). *Distributional Structure*. Word, 10(2-3), 146-162.

2. Firth, J. R. (1957). *A Synopsis of Linguistic Theory, 1930-1955*. Studies in Linguistic Analysis.

3. Mikolov, T., et al. (2013). *Efficient Estimation of Word Representations in Vector Space*. ICLR.  
   🔗 [https://arxiv.org/abs/1301.3781](https://arxiv.org/abs/1301.3781)

4. Pennington, J., Socher, R., & Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation*. EMNLP.  
   🔗 [https://nlp.stanford.edu/pubs/glove.pdf](https://nlp.stanford.edu/pubs/glove.pdf)

5. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

6. Radford, A., et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.  
   🔗 [https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)